In [ ]:
# =============================================================================
# kSZ²-21cm : Lightcone Simulation and Plotting
# =============================================================================

# =============================================================================
# CELL 1: Imports and Setup
# =============================================================================
import numpy as np
import matplotlib as mpl
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import py21cmfast as p21c
from py21cmfast import plotting
import os
from datetime import datetime

print(f"py21cmfast version: {p21c.__version__}")

# =============================================================================
# CELL 1a: Create Output Directory for Plots
# =============================================================================
plot_dir = "18March2026_kSZ2_21cm/plots"
if not os.path.exists(plot_dir):
    os.makedirs(plot_dir)
    print(f"Created directory: {plot_dir}")
else:
    print(f"Directory already exists: {plot_dir}")
print(f"All plots will be saved to: {os.path.abspath(plot_dir)}")

# =============================================================================
# CELL 1c: Define Parameters
# =============================================================================
user_params = p21c.UserParams(
    HII_DIM=128,
    BOX_LEN=800.0,
    USE_INTERPOLATION_TABLES=True,
    N_THREADS=16
)

z_min = 0.001
z_max = 20.0

# =============================================================================
# MULTI-SEED SETUP
# =============================================================================
RANDOM_SEEDS = list(range(1, 21))   # seeds 1, 2, 3, ..., 20
N_SEEDS      = len(RANDOM_SEEDS)
print(f"\n=== MULTI-SEED SETUP ===")
print(f"Seeds: {RANDOM_SEEDS}")
print(f"Total realisations: {N_SEEDS}")

# =============================================================================
default_astro = p21c.AstroParams()
print("\n=== USER PARAMETERS ===")
print(user_params)
print("\n=== DEFAULT COSMOLOGY ===")
print(p21c.CosmoParams())
print("\n=== DEFAULT ASTROPHYSICS ===")
print(p21c.AstroParams())
print("\n=== DEFAULT FLAGS ===")
print(p21c.FlagOptions())

py21cmfast version: 3.4.0
Directory already exists: 18March2026_kSZ2_21cm/plots
All plots will be saved to: /home/swanith/Desktop/Project2/Plots/kSZ_sqr_21cm_lightconev3/18March2026_kSZ2_21cm/plots

=== PARAMETER SCAN SETUP (kSZ²-21cm Analysis) ===

=== USER PARAMETERS ===
UserParams:
    BOX_LEN                 : 800.0
    DIM                     : 384
    FAST_FCOLL_TABLES       : False
    HII_DIM                 : 128
    HMF                     : 1
    KEEP_3D_VELOCITIES      : False
    MINIMIZE_MEMORY         : False
    NON_CUBIC_FACTOR        : 1.0
    NO_RNG                  : False
    N_THREADS               : 16
    PERTURB_ON_HIGH_RES     : False
    POWER_SPECTRUM          : 0
    USE_2LPT                : True
    USE_FFTW_WISDOM         : False
    USE_INTERPOLATION_TABLES: True
    USE_RELATIVE_VELOCITIES : False
    

=== DEFAULT COSMOLOGY ===
CosmoParams:
    OMb        : 0.04897468161869667
    OMm        : 0.30964144154550644
    POWER_INDEX: 0.9665
    SIGMA_8   

In [14]:
# =============================================================================
# CELL 1b: Standardized Plot Settings
# =============================================================================
plt.rcParams.update({
    # Font settings
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'font.size': 40,
    'axes.labelsize': 35,
    'axes.titlesize': 40,
    'xtick.labelsize': 30,
    'ytick.labelsize': 30,
    'legend.fontsize': 30,
    'figure.titlesize': 20,
    
    # Professional ticks
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 6,
    'ytick.major.size': 6,
    'xtick.minor.size': 3,
    'ytick.minor.size': 3,
    'xtick.top': True,
    'ytick.right': True,
    
    # Line and axes
    'axes.linewidth': 1.0,
    'lines.linewidth': 1.8,
    
    # Figure
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

mpl.rcParams['xtick.minor.visible'] = True
mpl.rcParams['ytick.minor.visible'] = True

print("✓ Plot settings applied")

✓ Plot settings applied


In [ ]:
# =============================================================================
# CELL 2: Run Lightcone Simulations for All Seeds (with Caching)
# =============================================================================

import time

print("\n" + "="*70)
print("RUNNING LIGHTCONE SIMULATIONS FOR kSZ²-21cm ANALYSIS")
print("="*70)

# Create cache directory
cache_dir = "18March2026_kSZ2_21cm/cache"
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)
    print(f"Created cache directory: {cache_dir}")
else:
    print(f"Cache directory exists: {cache_dir}")

# Use default astrophysical parameters
astro_params = p21c.AstroParams()

print(f"\nAstrophysical Parameters:")
print(f"  HII_EFF_FACTOR = {astro_params.HII_EFF_FACTOR}")
print(f"  ION_Tvir_MIN = {astro_params.ION_Tvir_MIN:.3f} (log10 K) = {10**astro_params.ION_Tvir_MIN:.2e} K")
print(f"\nRedshift range: z = {z_min} → {z_max}")
print(f"Box size: {user_params.BOX_LEN} Mpc")
print(f"Resolution: {user_params.HII_DIM}³ cells")

# =============================================================================
# Loop over seeds
# =============================================================================

lightcones = {}   # {seed: lightcone}

for seed in RANDOM_SEEDS:

    print(f"\n{'='*60}")
    print(f"SEED {seed}  ({RANDOM_SEEDS.index(seed)+1}/{N_SEEDS})")
    print(f"{'='*60}")

    # Per-seed cache and save paths
    seed_cache_dir  = f"{cache_dir}/seed_{seed}"
    seed_save_path  = f"{cache_dir}/seed_{seed}/lightcone_seed{seed}.pkl"

    if not os.path.exists(seed_cache_dir):
        os.makedirs(seed_cache_dir)

    # ------------------------------------------------------------------
    # Load from cache if it exists
    # ------------------------------------------------------------------
    if os.path.exists(seed_save_path):
        print(f"  Found cached lightcone → loading from {seed_save_path}")
        import pickle
        with open(seed_save_path, 'rb') as f:
            lightcone = pickle.load(f)
        print(f"  ✓ Loaded seed {seed} from cache")

    # ------------------------------------------------------------------
    # Otherwise run the simulation and cache it
    # ------------------------------------------------------------------
    else:
        print(f"  No cache found → running simulation...")
        sim_start = time.time()

        try:
            lightcone = p21c.run_lightcone(
                redshift=z_min,
                max_redshift=z_max,
                lightcone_quantities=('brightness_temp', 'density',
                                      'xH_box', 'velocity'),
                user_params=user_params,
                astro_params=astro_params,
                random_seed=seed,
                direc=seed_cache_dir
            )

            sim_time = time.time() - sim_start
            print(f"  ✓ Simulation complete in {sim_time/60:.2f} min")

            # Save to pickle
            import pickle
            with open(seed_save_path, 'wb') as f:
                pickle.dump(lightcone, f)
            print(f"  ✓ Cached to {seed_save_path}")

        except Exception as e:
            print(f"  ✗ Simulation FAILED for seed {seed}: {e}")
            lightcone = None

    # ------------------------------------------------------------------
    # Store and print reionization stats
    # ------------------------------------------------------------------
    if lightcone is not None:
        lightcones[seed] = lightcone

        z_nodes   = lightcone.node_redshifts[::-1]
        x_e_nodes = 1.0 - lightcone.global_xH[::-1]

        try:
            z_10 = z_nodes[np.argmin(np.abs(x_e_nodes - 0.1))]
            z_50 = z_nodes[np.argmin(np.abs(x_e_nodes - 0.5))]
            z_90 = z_nodes[np.argmin(np.abs(x_e_nodes - 0.9))]
            print(f"  Reionization: z(10%)={z_10:.2f}, "
                  f"z(50%)={z_50:.2f}, z(90%)={z_90:.2f}, "
                  f"Δz={z_10-z_90:.2f}")
        except:
            print(f"  Could not compute reionization stats")
    else:
        print(f"  ✗ Seed {seed} skipped — lightcone is None")

print(f"\n{'='*70}")
print(f"✓ ALL SEEDS COMPLETE")
print(f"  Successful realisations: {len(lightcones)}/{N_SEEDS}")
print(f"  Seeds loaded: {list(lightcones.keys())}")
print(f"{'='*70}")



RUNNING LIGHTCONE SIMULATION FOR kSZ²-21cm ANALYSIS
Cache directory exists: 18March2026_kSZ2_21cm/cache

Astrophysical Parameters:
  HII_EFF_FACTOR = 30.0
  ION_Tvir_MIN = 4.699 (log10 K) = 5.00e+04 K

Redshift range: z = 0.001 → 20.0
Box size: 800.0 Mpc
Resolution: 128³ cells

Running lightcone simulation...

✓ Simulation complete!
  Time: 3.12 minutes (0.05 hours)
  Cache: 18March2026_kSZ2_21cm/cache
  Shape: (128, 128, 1754)
  Redshift range: [0.00, 20.05]

  Reionization Statistics:
    z(10% ionized) = 10.90
    z(50% ionized) = 8.02
    z(90% ionized) = 6.40
    Δz (10%→90%) = 4.50


GENERATING LIGHTCONE PLOTS

Plotting brightness_temp...
  ✓ Saved: brightness_temp_lightcone

Plotting xH_box...
  ✓ Saved: xH_box_lightcone

Plotting density...
  ✓ Saved: density_lightcone

Plotting velocity...
  ✓ Saved: velocity_lightcone

✓ LIGHTCONE PLOTTING COMPLETE!


In [ ]:
# =============================================================================
# CELL 3: Reionization History Analysis (All Seeds)
# =============================================================================
print("\n" + "="*70)
print("REIONIZATION HISTORY ANALYSIS")
print("="*70)

if len(lightcones) > 0:

    # ==========================================================================
    # Collect reionization histories across all seeds
    # ==========================================================================
    all_z_nodes    = {}   # {seed: z_nodes}
    all_x_e_nodes  = {}   # {seed: x_e_nodes}
    all_xHI_nodes  = {}   # {seed: xHI_nodes}

    for seed, lc in lightcones.items():
        all_z_nodes[seed]   = lc.node_redshifts[::-1]
        all_x_e_nodes[seed] = 1.0 - lc.global_xH[::-1]
        all_xHI_nodes[seed] = lc.global_xH[::-1]

    # Common redshift grid for mean (interpolate all seeds onto it)
    z_common   = np.linspace(z_max, z_min, 500)
    xe_interp  = np.array([
        np.interp(z_common, all_z_nodes[s][::-1], all_x_e_nodes[s][::-1])
        for s in lightcones.keys()
    ])
    xHI_interp = np.array([
        np.interp(z_common, all_z_nodes[s][::-1], all_xHI_nodes[s][::-1])
        for s in lightcones.keys()
    ])

    xe_mean   = np.mean(xe_interp,  axis=0)
    xe_std    = np.std(xe_interp,   axis=0)
    xHI_mean  = np.mean(xHI_interp, axis=0)
    xHI_std   = np.std(xHI_interp,  axis=0)

    # Print mean reionization redshifts
    z_xe_half_mean = np.interp(0.5, xe_mean[::-1], z_common[::-1])
    print(f"Mean z(x_e = 0.5) across seeds: z = {z_xe_half_mean:.2f}")
    for seed in lightcones.keys():
        z_half = np.interp(0.5, all_x_e_nodes[seed][::-1],
                           all_z_nodes[seed][::-1])
        print(f"  Seed {seed:3d}: z(x_e=0.5) = {z_half:.2f}")

    # Color map for seeds
    cmap_seeds = plt.cm.plasma(np.linspace(0.1, 0.9, N_SEEDS))

    # ==========================================================================
    # PLOT 3a: Ionization Fraction x_e vs Redshift
    # ==========================================================================
    fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

    # Individual seed lines
    for i, (seed, lc) in enumerate(lightcones.items()):
        ax.plot(all_z_nodes[seed], all_x_e_nodes[seed],
                color=cmap_seeds[i], lw=1.0, alpha=0.4)

    # Mean + std band
    ax.fill_between(z_common, xe_mean - xe_std, xe_mean + xe_std,
                    color='darkblue', alpha=0.2, label=r'$\pm 1\sigma$')
    ax.plot(z_common, xe_mean,
            color='darkblue', lw=2.5, label=f'Mean ({N_SEEDS} seeds)')

    # Mark x_e = 0.5 on mean
    ax.axhline(0.5, color='gray', linestyle='--', lw=1, alpha=0.7)
    ax.axvline(z_xe_half_mean, color='gray', linestyle='--', lw=1, alpha=0.7)
    ax.text(z_xe_half_mean, 0.52,
            rf'$z_{{x_e=0.5}} = {z_xe_half_mean:.2f}$',
            fontsize=13, ha='right', va='bottom',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    ax.set_xlabel(r'Redshift $z$', fontsize=14)
    ax.set_ylabel(r'Ionization Fraction $x_e$', fontsize=14)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(loc='best', fontsize=12)
    ax.invert_xaxis()
    ax.grid(True, alpha=0.3, linestyle='--')

    plot_name = "reionization_history_xe"
    fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
    ax.set_title('Reionization History: Ionization Fraction',
                 fontsize=20, fontweight='bold')
    fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_name}")
    plt.close(fig)

    # ==========================================================================
    # PLOT 3b: Neutral Fraction xHI vs Redshift
    # ==========================================================================
    fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

    # Individual seed lines
    for i, (seed, lc) in enumerate(lightcones.items()):
        ax.plot(all_z_nodes[seed], all_xHI_nodes[seed],
                color=cmap_seeds[i], lw=1.0, alpha=0.4)

    # Mean + std band
    ax.fill_between(z_common, xHI_mean - xHI_std, xHI_mean + xHI_std,
                    color='darkred', alpha=0.2, label=r'$\pm 1\sigma$')
    ax.plot(z_common, xHI_mean,
            color='darkred', lw=2.5, label=f'Mean ({N_SEEDS} seeds)')

    ax.set_xlabel(r'Redshift $z$', fontsize=14)
    ax.set_ylabel(r'Neutral Fraction $x_{\rm HI}$', fontsize=14)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(loc='best', fontsize=12)
    ax.invert_xaxis()
    ax.grid(True, alpha=0.3, linestyle='--')

    ax.text(0.05, 0.95,
            f'HII_EFF_FACTOR = {astro_params.HII_EFF_FACTOR:.1f}\n'
            f'ION_Tvir_MIN = {astro_params.ION_Tvir_MIN:.2f} (log10 K)\n'
            f'N seeds = {N_SEEDS}',
            transform=ax.transAxes, fontsize=13,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plot_name = "reionization_history_xHI"
    fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
    ax.set_title('Reionization History: Neutral Fraction',
                 fontsize=20, fontweight='bold')
    fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_name}")
    plt.close(fig)

    print("\n✓ REIONIZATION HISTORY PLOTS COMPLETE")

else:
    print("\n✗ Skipping — no lightcones available")


REIONIZATION HISTORY ANALYSIS
Redshift at x_e = 0.5: z = 8.06
✓ Saved: reionization_history_xe
✓ Saved: reionization_history_xHI


In [ ]:
# =============================================================================
# CELL 4: Optical Depth Calculations (All Seeds)
# =============================================================================

print("\n" + "="*70)
print("OPTICAL DEPTH CALCULATION")
print("="*70)

if len(lightcones) > 0:

    # Physical constants (same for all seeds)
    c_km_s       = 2.998e5
    h            = 0.6766
    H0           = 100 * h
    Omega_b      = 0.04897468161869667
    Omega_m      = 0.30964144154550644
    rho_crit_p_cm3 = 1.88e-29 * h**2 / (1.67e-24)
    n_H0_cm3     = Omega_b * rho_crit_p_cm3
    sigma_T_cm2  = 6.65e-25
    cm_per_Mpc   = 3.086e24
    n_e0_Mpc3    = n_H0_cm3 * cm_per_Mpc**3
    sigma_T_Mpc2 = sigma_T_cm2 / cm_per_Mpc**2
    prefactor    = n_e0_Mpc3 * sigma_T_Mpc2

    print(f"\nPhysical constants:")
    print(f"  n_H0     = {n_H0_cm3:.6e} cm^-3")
    print(f"  σ_T      = {sigma_T_cm2:.6e} cm^2")
    print(f"  Prefactor= {prefactor:.6e} Mpc^-1")

    # ==========================================================================
    # Compute optical depth per seed and store
    # ==========================================================================

    # These are needed by downstream cells (Cell 6 kSZ integration)
    # so we store them in dicts keyed by seed
    tau_results = {}   # {seed: {'z_mid': , 'tau': , 'tau_total': ,
                       #         'ds_Mpc': , 'z_mid': , 'x_e_mid': }}

    for seed, lc in lightcones.items():

        # Redshift and distance axes
        red_axis = lc.lightcone_redshifts
        pos_axis = lc.lightcone_distances

        # Trim to z <= z_max
        ind_z    = np.where(red_axis <= z_max)[0]
        red_axis = red_axis[ind_z]
        pos_axis = pos_axis[ind_z]

        # Ionization history
        z_nodes_sorted  = lc.node_redshifts[::-1]
        xHI_nodes_sorted= lc.global_xH[::-1]
        x_e_nodes_sorted= 1.0 - xHI_nodes_sorted

        # Interpolate x_e onto lightcone redshift grid
        x_e_interp = np.interp(red_axis, z_nodes_sorted, x_e_nodes_sorted)

        # Distance elements
        ds_Mpc = np.asarray(np.diff(pos_axis), dtype=np.float64)

        # Midpoint values
        z_mid   = 0.5 * (red_axis[:-1] + red_axis[1:])
        x_e_mid = 0.5 * (x_e_interp[:-1] + x_e_interp[1:])

        # dτ and cumulative τ
        dtau      = prefactor * x_e_mid * (1.0 + z_mid)**2 * ds_Mpc
        tau       = np.cumsum(dtau)
        tau_total = tau[-1]

        tau_results[seed] = {
            'red_axis' : red_axis,
            'z_mid'    : z_mid,
            'x_e_mid'  : x_e_mid,
            'ds_Mpc'   : ds_Mpc,
            'tau'      : tau,
            'tau_total': tau_total,
        }

        print(f"  Seed {seed:3d}: τ_total = {tau_total:.6f}")

    # Summary stats across seeds
    tau_totals = np.array([tau_results[s]['tau_total'] for s in lightcones])
    print(f"\n  Mean τ = {tau_totals.mean():.6f} ± {tau_totals.std():.6f}")

    # ==========================================================================
    # Common redshift grid for mean τ curve
    # ==========================================================================
    z_common = tau_results[RANDOM_SEEDS[0]]['z_mid']   # all seeds share same grid
    tau_matrix = np.array([tau_results[s]['tau'] for s in lightcones])
    tau_mean   = np.mean(tau_matrix, axis=0)
    tau_std    = np.std(tau_matrix,  axis=0)

    # ==========================================================================
    # PLOT 4a: Cumulative Optical Depth τ vs z (all seeds + mean)
    # ==========================================================================
    cmap_seeds = plt.cm.plasma(np.linspace(0.1, 0.9, N_SEEDS))

    fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

    # Individual seed lines
    for i, seed in enumerate(lightcones):
        ax.plot(tau_results[seed]['z_mid'], tau_results[seed]['tau'],
                color=cmap_seeds[i], lw=1.0, alpha=0.4)

    # Mean + std band
    ax.fill_between(z_common, tau_mean - tau_std, tau_mean + tau_std,
                    color='darkgreen', alpha=0.2, label=r'$\pm 1\sigma$')
    ax.plot(z_common, tau_mean,
            color='darkgreen', lw=2.5, label=f'Mean ({N_SEEDS} seeds)')

    ax.set_xlabel(r'Redshift $z$', fontsize=14)
    ax.set_ylabel(r'Cumulative Optical Depth $\tau(<z)$', fontsize=14)
    ax.legend(loc='best', fontsize=12)
    ax.invert_xaxis()
    ax.grid(True, alpha=0.3, linestyle='--')

    ax.text(0.05, 0.95,
            f'Mean τ = {tau_totals.mean():.4f} ± {tau_totals.std():.4f}\n'
            f'HII_EFF_FACTOR = {astro_params.HII_EFF_FACTOR:.1f}\n'
            f'N seeds = {N_SEEDS}',
            transform=ax.transAxes, fontsize=12,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plot_name = "tau_vs_z"
    fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
    ax.set_title('Cumulative Optical Depth vs Redshift',
                 fontsize=20, fontweight='bold')
    fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
    print(f"\n✓ Saved: {plot_name}")
    plt.close(fig)

    print("\n✓ OPTICAL DEPTH ANALYSIS COMPLETE!")

else:
    print("\n✗ Skipping — no lightcones available")


OPTICAL DEPTH CALCULATION

Physical constants:
  n_H0 = 2.523928e-07 cm^-3
  σ_T = 6.650000e-25 cm^2
  Prefactor = 5.179580e-07 Mpc^-1

Optical Depth Results:
  Redshift range: [0.00, 19.97]
  Mean ds = 6.250 Mpc
  Total optical depth τ = 0.075890
✓ Saved: tau_vs_z

✓ OPTICAL DEPTH ANALYSIS COMPLETE!


In [ ]:
# =============================================================================
# CELL 5: Compute kSZ Integrand with Visibility Function (All Seeds)
# kSZ integrand = (1 + δ) × x_e × v_z / c × e^(-τ(z))
# Skips computation for seeds whose Cell 6 kSZ map cache already exists
# =============================================================================

print("\n" + "="*70)
print("COMPUTING kSZ INTEGRAND WITH VISIBILITY FUNCTION")
print("="*70)

if len(lightcones) > 0 and len(tau_results) > 0:

    c_Mpc_s = 299792.458 / 3.08567758e19
    print(f"Speed of light: c = {c_Mpc_s:.6e} Mpc/s")

    kSZ_integrands = {}   # {seed: kSZ_integrand 3D array}

    for seed, lc in lightcones.items():

        print(f"\n--- Seed {seed} ---")

        # ------------------------------------------------------------------
        # Skip if Cell 6 kSZ map cache already exists — integrand not needed
        # ------------------------------------------------------------------
        map_path = f"{cache_dir}/kSZ_maps/kSZ_map_z{z_obs:.1f}_seed{seed}.npy"
        if os.path.exists(map_path):
            print(f"  Cell 6 cache exists → skipping integrand computation")
            kSZ_integrands[seed] = None   # placeholder so downstream knows seed exists
            continue

        # ------------------------------------------------------------------
        # Compute integrand
        # ------------------------------------------------------------------
        tr = tau_results[seed]

        red_axis_full = np.asarray(lc.lightcone_redshifts)
        ind_z         = np.where(red_axis_full <= z_max)[0]

        density_1plus = 1 + np.asarray(lc.density[:, :, ind_z])
        x_e_3D        = 1 - np.asarray(lc.xH_box[:, :, ind_z])
        v_los_Mpc_s   = np.asarray(lc.velocity[:, :, ind_z])

        red_axis_array = np.asarray(tr['red_axis'], dtype=np.float64)
        tau_array      = np.asarray(tr['tau'],      dtype=np.float64)
        z_mid_array    = np.asarray(tr['z_mid'],    dtype=np.float64)

        tau_extended  = np.concatenate([[0.0], tau_array])
        z_extended    = np.concatenate([[red_axis_array[0]], z_mid_array])
        tau_at_lc     = np.asarray(
            np.interp(red_axis_array, z_extended, tau_extended),
            dtype=np.float64
        )

        visibility    = np.exp(-tau_at_lc)
        visibility_3D = visibility[None, None, :]

        kSZ_integrand = (density_1plus * x_e_3D
                         * v_los_Mpc_s / c_Mpc_s
                         * visibility_3D)

        kSZ_integrands[seed] = kSZ_integrand

        print(f"  Shape : {kSZ_integrand.shape}")
        print(f"  Mean  : {kSZ_integrand.mean():.4e}")
        print(f"  Std   : {kSZ_integrand.std():.4e}")
        print(f"  RMS   : {np.sqrt(np.mean(kSZ_integrand**2)):.4e}")

    n_computed = sum(1 for v in kSZ_integrands.values() if v is not None)
    n_skipped  = N_SEEDS - n_computed
    print(f"\n✓ Integrand computed: {n_computed} seeds")
    print(f"  Skipped (Cell 6 cache found): {n_skipped} seeds")

    # ==========================================================================
    # PLOT 5a: kSZ Integrand Lightcone (random seed that was actually computed)
    # ==========================================================================

    # Only plot from seeds that were computed (not skipped)
    computed_seeds = [s for s, v in kSZ_integrands.items() if v is not None]

    if len(computed_seeds) > 0:
        seed_to_plot  = int(np.random.choice(computed_seeds))
        print(f"\nRandomly selected seed for plot: {seed_to_plot}")

        lc            = lightcones[seed_to_plot]
        kSZ_integrand = kSZ_integrands[seed_to_plot]

        red_axis_full = np.asarray(lc.lightcone_redshifts)
        ind_z         = np.where(red_axis_full <= z_max)[0]

        fig, ax = plt.subplots(1, 1, figsize=(12, 5), constrained_layout=True)

        slice_2D = kSZ_integrand[:, :, kSZ_integrand.shape[2]//2]
        x_extent = float(np.asarray(lc.lightcone_distances[ind_z].max()))
        y_extent = float(user_params.BOX_LEN)

        im = ax.imshow(slice_2D.T,
                       extent=[0, x_extent, 0, y_extent],
                       aspect='auto', cmap='seismic', origin='lower')

        vmax = np.percentile(np.abs(kSZ_integrand), 99)
        im.set_clim(-vmax, vmax)

        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label(r'kSZ Integrand [dimensionless]', fontsize=16)

        ax.set_xlabel('Comoving Distance [Mpc]', fontsize=16)
        ax.set_ylabel('Comoving Distance [Mpc]', fontsize=16)

        ax2 = ax.twiny()
        ax2.set_xlim(ax.get_xlim())
        distance_ticks     = ax.get_xticks()
        lc_distances_float = np.asarray(lc.lightcone_distances, dtype=np.float64)
        lc_redshifts_float = np.asarray(lc.lightcone_redshifts, dtype=np.float64)
        z_ticks = np.interp(distance_ticks, lc_distances_float, lc_redshifts_float)
        ax2.set_xticks(distance_ticks)
        ax2.set_xticklabels([f'{z:.1f}' for z in z_ticks])
        ax2.set_xlabel('Redshift $z$', fontsize=16)

        ax.text(0.02, 0.98,
                f'seed={seed_to_plot} | '
                f'HII_EFF_FACTOR={astro_params.HII_EFF_FACTOR:.1f}, '
                f'ION_Tvir_MIN={astro_params.ION_Tvir_MIN:.2f} (log10 K)',
                transform=ax.transAxes, fontsize=12, fontweight='bold',
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        plot_name = f"kSZ_integrand_with_visibility_seed{seed_to_plot}"
        fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
        fig.suptitle(
            r'kSZ Integrand: $(1+\delta)\times x_e\times v_z/c\times e^{-\tau(z)}$'
            f'  [seed={seed_to_plot}]',
            fontsize=18, fontweight='bold')
        fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
        print(f"✓ Saved: {plot_name}")
        plt.close(fig)

        print("\n✓ kSZ INTEGRAND PLOTTING COMPLETE!")

    else:
        print("\n  All seeds loaded from Cell 6 cache — no integrand plot generated")

else:
    print("\n✗ Skipping — lightcones or tau_results not available")


COMPUTING kSZ INTEGRAND WITH VISIBILITY FUNCTION
Speed of light: c = 9.715612e-15 Mpc/s

3D field shapes: (128, 128, 1753)
Redshift range: [0.00, 19.97]

Optical depth range at lightcone:
  τ range: [0.000000, 0.075890]
  τ dtype: float64
  e^(-τ) range: [0.926918, 1.000000]

kSZ INTEGRAND (with visibility) STATISTICS:
  Mean: -6.5905e-07
  Std:  2.1631e-03
  Min:  -1.0796e-01
  Max:  1.1790e-01
  RMS:  2.1631e-03

✓ kSZ INTEGRAND CALCULATION COMPLETE

GENERATING kSZ INTEGRAND PLOT
✓ Saved: kSZ_integrand_with_visibility

✓ kSZ INTEGRAND PLOTTING COMPLETE!


In [ ]:
# =============================================================================
# CELL 6: Compute Line-of-Sight Integrated kSZ Maps for All Seeds
# kSZ(z_obs=5) = ∫ from z_start to z=5 of [n_e0 σ_T (1/a²) (1+δ) x_e v_z/c e^(-τ) ds]
# =============================================================================

print("\n" + "="*70)
print("LINE-OF-SIGHT kSZ MAP INTEGRATION AT z_obs = 5")
print("="*70)

if len(lightcones) > 0:

    # Create directory for storing kSZ maps
    kSZ_maps_dir = f"{cache_dir}/kSZ_maps"
    if not os.path.exists(kSZ_maps_dir):
        os.makedirs(kSZ_maps_dir)
        print(f"Created directory: {kSZ_maps_dir}")
    else:
        print(f"Directory exists: {kSZ_maps_dir}")

    # Physical constants in CGS
    print(f"\n=== PHYSICAL CONSTANTS (CGS) ===")
    c_cm_s      = 3.0e10
    sigma_T_cm2 = 6.6525e-25
    n_e0_cm3    = 2.06e-7
    Mpc_to_cm   = 3.0857e24

    print(f"c = {c_cm_s:.2e} cm/s")
    print(f"σ_T = {sigma_T_cm2:.4e} cm²")
    print(f"n_e0 = {n_e0_cm3:.4e} cm⁻³")
    print(f"1 Mpc = {Mpc_to_cm:.4e} cm")

    prefactor_cgs = n_e0_cm3 * sigma_T_cm2 * c_cm_s
    print(f"Prefactor n_e0 × σ_T × c = {prefactor_cgs:.4e} s⁻¹")

    z_obs = 5.0
    print(f"\nz_obs = {z_obs:.1f} (end of reionization)")

    # ==========================================================================
    # Loop over seeds
    # ==========================================================================

    kSZ_maps = {}   # {seed: kSZ_map_2D}

    for seed, lc in lightcones.items():

        print(f"\n--- Seed {seed} ---")

        map_path = f"{kSZ_maps_dir}/kSZ_map_z{z_obs:.1f}_seed{seed}.npy"

        # ------------------------------------------------------------------
        # CASE 1: Cell 6 cache exists → load directly, skip everything
        # ------------------------------------------------------------------
        if os.path.exists(map_path):
            print(f"  Found cached kSZ map → loading")
            kSZ_maps[seed] = np.load(map_path)
            print(f"  ✓ Loaded | RMS: {np.sqrt(np.mean(kSZ_maps[seed]**2)):.4e}")
            continue

        # ------------------------------------------------------------------
        # CASE 2: Cell 5 integrand available in memory → use it
        # ------------------------------------------------------------------
        integrand_available = (
            'kSZ_integrands' in dir()
            and seed in kSZ_integrands
            and kSZ_integrands[seed] is not None
        )

        if not integrand_available:
            print(f"  ✗ No integrand in memory and no cached map — skipping seed {seed}")
            print(f"    Re-run Cell 5 to compute the integrand first")
            continue

        # ------------------------------------------------------------------
        # CASE 3: Compute kSZ map from integrand
        # ------------------------------------------------------------------
        print(f"  Computing from integrand...")

        tr     = tau_results[seed]
        red_axis = np.asarray(tr['red_axis'], dtype=np.float64)
        ds_Mpc   = np.asarray(tr['ds_Mpc'],   dtype=np.float64)
        z_mid    = np.asarray(tr['z_mid'],    dtype=np.float64)
        ds_cm    = ds_Mpc * Mpc_to_cm

        a                = 1.0 / (1.0 + red_axis)
        a_squared        = a**2
        a_squared_mid    = 0.5 * (a_squared[:-1] + a_squared[1:])
        a_squared_mid_3D = a_squared_mid[None, None, :]

        kSZ_int      = kSZ_integrands[seed]
        kSZ_int_mid  = 0.5 * (kSZ_int[:, :, :-1] + kSZ_int[:, :, 1:])
        kSZ_int_full = ((prefactor_cgs / a_squared_mid_3D)
                        * kSZ_int_mid
                        * (ds_cm / c_cm_s)[None, None, :])

        idx_integrate = np.where(z_mid >= z_obs)[0]
        print(f"  Integration: z = {z_mid[idx_integrate].max():.2f} → "
              f"{z_mid[idx_integrate].min():.2f} ({len(idx_integrate)} slices)")

        import time
        t0      = time.time()
        kSZ_map = np.sum(kSZ_int_full[:, :, idx_integrate], axis=2)
        print(f"  Computed in {time.time()-t0:.2f}s")

        np.save(map_path, kSZ_map)
        print(f"  ✓ Saved to {map_path}")

        kSZ_maps[seed] = kSZ_map
        print(f"  Mean: {kSZ_map.mean():.4e} | "
              f"RMS: {np.sqrt(np.mean(kSZ_map**2)):.4e} | "
              f"Std: {kSZ_map.std():.4e}")

    # ==========================================================================
    # Summary
    # ==========================================================================
    print(f"\n✓ kSZ MAPS READY: {len(kSZ_maps)}/{N_SEEDS} SEEDS")

    if len(kSZ_maps) > 0:
        rms_all = np.array([np.sqrt(np.mean(kSZ_maps[s]**2)) for s in kSZ_maps])
        print(f"  RMS across seeds : {rms_all.mean():.4e} ± {rms_all.std():.4e}")
        print(f"  Map dimensions   : {kSZ_maps[RANDOM_SEEDS[0]].shape[0]} × "
              f"{kSZ_maps[RANDOM_SEEDS[0]].shape[1]} pixels")
        print(f"  Physical size    : {user_params.BOX_LEN:.1f} × "
              f"{user_params.BOX_LEN:.1f} Mpc²")
        print(f"  Pixel size       : "
              f"{user_params.BOX_LEN/kSZ_maps[RANDOM_SEEDS[0]].shape[0]:.2f} Mpc")

else:
    print("\n✗ Skipping — no lightcones available")

print("\n" + "="*70)


LINE-OF-SIGHT kSZ MAP INTEGRATION AT z_obs = 5
Created directory: 18March2026_kSZ2_21cm/cache/kSZ_maps

=== PHYSICAL CONSTANTS (CGS) ===
c = 3.00e+10 cm/s
σ_T = 6.6525e-25 cm²
n_e0 = 2.0600e-07 cm⁻³
1 Mpc = 3.0857e+24 cm

Prefactor n_e0 × σ_T × c = 4.1112e-21 s⁻¹

=== OBSERVER REDSHIFT ===
z_obs = 5.0 (end of reionization)
ds in cm: 1.9286e+25 - 1.9286e+25 cm

=== PREPARING 3D FIELDS ===
kSZ integrand shape: (128, 128, 1752)
Full lightcone: z = 19.97 to z = 0.0010

=== COMPUTING kSZ MAP AT z_obs = 5.0 ===
Integration range: z = 19.93 to z = 5.01
Number of slices integrated: 481

✓ kSZ MAP COMPUTATION COMPLETE
  Computation time: 0.02 seconds
  Saved to: /home/swanith/Desktop/Project2/Plots/kSZ_sqr_21cm_lightconev3/18March2026_kSZ2_21cm/cache/kSZ_maps/kSZ_map_z5.0.npy

=== kSZ MAP STATISTICS ===
Observer redshift: z = 5.0
Integrated slices: 481
Map dimensions: 128 × 128 pixels
Physical size: 800.0 × 800.0 Mpc²
Pixel size: 6.25 Mpc

Statistical measures:
  Mean: 1.9065e-07
  RMS: 8.2898

In [ ]:
# =============================================================================
# CELL 7: Compute kSZ²-21cm Cross-Correlation Power Spectra (All Seeds)
# Cross-correlating integrated kSZ² map with 21cm maps at various redshifts
# =============================================================================

print("\n" + "="*70)
print("COMPUTING kSZ²-21cm CROSS-CORRELATION POWER SPECTRA")
print("="*70)

if len(lightcones) > 0:

    # Map properties (same for all seeds)
    npix_side    = user_params.HII_DIM
    box_size_Mpc = float(user_params.BOX_LEN)
    pix_size_Mpc = box_size_Mpc / npix_side
    pix_area     = pix_size_Mpc**2

    print(f"\n=== MAP PROPERTIES ===")
    print(f"Map size    : {npix_side} × {npix_side} pixels")
    print(f"Physical    : {box_size_Mpc:.1f} × {box_size_Mpc:.1f} Mpc²")
    print(f"Pixel size  : {pix_size_Mpc:.3f} Mpc/pixel")

    # k-space grid (same for all seeds)
    dk        = 2 * np.pi / (npix_side * pix_size_Mpc)
    kx        = np.fft.fftshift(np.fft.fftfreq(npix_side)) * npix_side * dk
    ky        = np.fft.fftshift(np.fft.fftfreq(npix_side)) * npix_side * dk
    kgrid     = np.sqrt(kx[:, None]**2 + ky[None, :]**2)
    k_bins    = np.logspace(np.log10(dk), np.log10(kgrid.max()*0.9), 35)
    k_centers = 0.5 * (k_bins[:-1] + k_bins[1:])

    print(f"\n=== k-SPACE GRID ===")
    print(f"dk          : {dk:.6f} Mpc⁻¹")
    print(f"k range     : [{kgrid.min():.6f}, {kgrid.max():.6f}] Mpc⁻¹")

    from astropy.cosmology import FlatLambdaCDM
    import time
    cosmo = FlatLambdaCDM(H0=67.77, Om0=0.3086)
    z_obs = 5.0

    # kSZ maps directory (Cell 6 cache)
    kSZ_maps_dir = f"{cache_dir}/kSZ_maps"

    cross_corr_results_all = {}

    for seed, lc in lightcones.items():

        print(f"\n{'='*60}")
        print(f"SEED {seed}  ({list(lightcones.keys()).index(seed)+1}/{N_SEEDS})")
        print(f"{'='*60}")

        # ------------------------------------------------------------------
        # CASE 1: Cross-corr cache exists → load directly
        # ------------------------------------------------------------------
        cc_cache = f"{cache_dir}/seed_{seed}/cross_corr_seed{seed}.npy"

        if os.path.exists(cc_cache):
            print(f"  Found cached cross-corr → loading")
            cross_corr_results_all[seed] = np.load(
                cc_cache, allow_pickle=True).item()
            print(f"  ✓ Loaded {len(cross_corr_results_all[seed])} redshifts")
            continue

        # ------------------------------------------------------------------
        # CASE 2: No cross-corr cache → need kSZ map
        # Try memory first, then Cell 6 cache, then fail gracefully
        # ------------------------------------------------------------------
        kSZ_map = None

        if 'kSZ_maps' in dir() and seed in kSZ_maps and kSZ_maps[seed] is not None:
            print(f"  kSZ map found in memory")
            kSZ_map = kSZ_maps[seed]
        else:
            map_path = f"{kSZ_maps_dir}/kSZ_map_z{z_obs:.1f}_seed{seed}.npy"
            if os.path.exists(map_path):
                print(f"  kSZ map not in memory → loading from Cell 6 cache")
                kSZ_map = np.load(map_path)
                if 'kSZ_maps' not in dir():
                    kSZ_maps = {}
                kSZ_maps[seed] = kSZ_map
                print(f"  ✓ Loaded kSZ map | RMS: {np.sqrt(np.mean(kSZ_map**2)):.4e}")
            else:
                print(f"  ✗ No kSZ map available for seed {seed} — skipping")
                print(f"    Re-run Cells 5 and 6 to generate the kSZ map")
                continue

        # ------------------------------------------------------------------
        # No high-pass filter applied — k→ℓ conversion (comoving vs
        # angular diameter distance) needs to be settled before filtering
        # ------------------------------------------------------------------
        kSZ_map_filtered = kSZ_map
        print(f"  kSZ map used as-is (no high-pass filter)")
        print(f"  RMS: {kSZ_map.std():.4e}")

        # Square map
        kSZ2_map          = kSZ_map_filtered**2
        kSZ2_map_centered = kSZ2_map - np.mean(kSZ2_map)
        fft_kSZ2_shifted  = np.fft.fftshift(np.fft.fft2(kSZ2_map_centered))
        auto_kSZ2_ps2d    = np.abs(fft_kSZ2_shifted)**2 * pix_area / npix_side**2
        print(f"  kSZ² RMS: {np.sqrt(np.mean(kSZ2_map**2)):.4e}")

        # ------------------------------------------------------------------
        # Loop over node redshifts
        # ------------------------------------------------------------------
        node_redshifts     = np.asarray(lc.node_redshifts[::-1])
        cross_corr_results = {}
        loop_start         = time.time()

        for i, z_21cm in enumerate(node_redshifts):

            lc_redshifts       = np.asarray(lc.lightcone_redshifts, dtype=np.float64)
            idx_closest        = np.argmin(np.abs(lc_redshifts - z_21cm))
            z_actual           = lc_redshifts[idx_closest]
            T21_slice          = np.asarray(lc.brightness_temp[:, :, idx_closest])
            T21_slice_centered = T21_slice - np.mean(T21_slice)
            fft_T21_shifted    = np.fft.fftshift(np.fft.fft2(T21_slice_centered))

            cross_ps2d    = (np.real(np.conj(fft_kSZ2_shifted) * fft_T21_shifted)
                             * pix_area / npix_side**2)
            auto_T21_ps2d = np.abs(fft_T21_shifted)**2 * pix_area / npix_side**2

            C_cross_1d            = np.zeros(len(k_centers))
            C_cross_1d_err_sample = np.zeros(len(k_centers))
            C_cross_1d_err_cosmic = np.zeros(len(k_centers))
            C_cross_1d_err_total  = np.zeros(len(k_centers))
            P_kSZ2_1d             = np.zeros(len(k_centers))
            P_T21_1d              = np.zeros(len(k_centers))
            n_modes               = np.zeros(len(k_centers))

            for j in range(len(k_centers)):
                mask  = (kgrid >= k_bins[j]) & (kgrid < k_bins[j+1])
                n_pix = np.sum(mask)

                if n_pix > 0:
                    cross_values             = cross_ps2d[mask]
                    C_cross_1d[j]            = np.mean(cross_values)
                    C_cross_1d_err_sample[j] = np.std(cross_values) / np.sqrt(n_pix)
                    P_kSZ2_1d[j]             = np.mean(auto_kSZ2_ps2d[mask])
                    P_T21_1d[j]              = np.mean(auto_T21_ps2d[mask])
                    k_volume                 = (box_size_Mpc / (2*np.pi))**3
                    n_modes[j]               = (4 * np.pi * k_centers[j]**2
                                                * k_volume
                                                * (k_bins[j+1] - k_bins[j]))
                    if n_modes[j] > 0:
                        C_cross_1d_err_cosmic[j] = (np.abs(C_cross_1d[j])
                                                     / np.sqrt(n_modes[j]))
                    else:
                        C_cross_1d_err_cosmic[j] = np.nan
                    C_cross_1d_err_total[j] = np.sqrt(
                        C_cross_1d_err_sample[j]**2 + C_cross_1d_err_cosmic[j]**2)
                else:
                    C_cross_1d[j] = C_cross_1d_err_sample[j] = np.nan
                    C_cross_1d_err_cosmic[j] = C_cross_1d_err_total[j] = np.nan
                    P_kSZ2_1d[j]  = P_T21_1d[j] = np.nan

            cross_corr_results[z_21cm] = {
                'k_centers'            : k_centers,
                'C_cross_1d'           : C_cross_1d,
                'C_cross_1d_err_sample': C_cross_1d_err_sample,
                'C_cross_1d_err_cosmic': C_cross_1d_err_cosmic,
                'C_cross_1d_err_total' : C_cross_1d_err_total,
                'n_modes'              : n_modes,
                'P_kSZ2_1d'            : P_kSZ2_1d,
                'P_T21_1d'             : P_T21_1d,
                'z_actual'             : z_actual,
                'idx_closest'          : idx_closest,
                'kSZ2_rms'             : np.sqrt(np.mean(kSZ2_map**2)),
                'T21_rms'              : np.sqrt(np.mean(T21_slice**2)),
                'T21_mean'             : np.mean(T21_slice)
            }

            if (i+1) % 10 == 0 or i == 0 or i == len(node_redshifts)-1:
                elapsed  = time.time() - loop_start
                eta      = (elapsed/(i+1)) * (len(node_redshifts)-(i+1))
                valid    = ~np.isnan(C_cross_1d)
                sign_str = ("+" if C_cross_1d[len(C_cross_1d)//2] > 0
                            else "-") if np.any(valid) else "?"
                print(f"  [{i+1:3d}/{len(node_redshifts)}] z={z_21cm:.3f} "
                      f"sign={sign_str} "
                      f"T21_mean={np.mean(T21_slice):.2f} mK | "
                      f"ETA: {eta:.1f}s")

        total_time = time.time() - loop_start
        print(f"\n  ✓ Seed {seed} complete in {total_time/60:.2f} min")

        np.save(cc_cache, cross_corr_results)
        print(f"  ✓ Cached to {cc_cache}")

        cross_corr_results_all[seed] = cross_corr_results

    print(f"\n{'='*70}")
    print(f"✓ CROSS-CORRELATIONS READY: {len(cross_corr_results_all)}/{N_SEEDS} SEEDS")

    # ==========================================================================
    # Error budget summary (averaged across all seeds)
    # ==========================================================================
    print(f"\n=== ERROR BUDGET SUMMARY (averaged across seeds) ===")

    all_sample_err, all_cosmic_err, all_total_err = [], [], []

    for seed, ccr in cross_corr_results_all.items():
        for z_21cm, res in ccr.items():
            C     = res['C_cross_1d']
            valid = ~np.isnan(C) & (C != 0)
            if np.any(valid):
                all_sample_err.extend(
                    (res['C_cross_1d_err_sample'][valid] / np.abs(C[valid])).tolist())
                all_cosmic_err.extend(
                    (res['C_cross_1d_err_cosmic'][valid] / np.abs(C[valid])).tolist())
                all_total_err.extend(
                    (res['C_cross_1d_err_total'][valid]  / np.abs(C[valid])).tolist())

    if len(all_sample_err) > 0:
        print(f"  Sample variance (mean): {np.nanmean(all_sample_err)*100:.1f}%")
        print(f"  Cosmic variance (mean): {np.nanmean(all_cosmic_err)*100:.1f}%")
        print(f"  Total  variance (mean): {np.nanmean(all_total_err)*100:.1f}%")

else:
    print("\n✗ Skipping — no lightcones available")

print("\n" + "="*70)


COMPUTING kSZ²-21cm CROSS-CORRELATION POWER SPECTRA

Number of node redshifts: 155
Redshift range: [0.0010, 20.1280]

=== MAP PROPERTIES ===
Map size: 128 × 128 pixels
Physical size: 800.0 × 800.0 Mpc²
Pixel size: 6.250 Mpc/pixel

=== k-SPACE GRID ===
dk (fundamental): 0.007854 Mpc⁻¹
k range: [0.000000, 0.710861] Mpc⁻¹

=== PREPARING kSZ² MAP (z_obs = 5) ===
kSZ² map statistics:
  Mean (before centering): 6.8721e-11
  RMS: 1.4273e-10
  Std: 1.2510e-10

=== COMPUTING CROSS-CORRELATIONS ===
Cross-correlating kSZ² map (z=5) with 21cm maps at various redshifts
  [  1/155] z_21cm=0.0010 (Δz=0.0000): Sign=-, T21_mean=0.00 mK | ETA: 519.6s
  [ 10/155] z_21cm=0.1963 (Δz=0.0002): Sign=-, T21_mean=0.00 mK | ETA: 428.6s
  [ 20/155] z_21cm=0.4583 (Δz=0.0000): Sign=-, T21_mean=0.00 mK | ETA: 398.3s
  [ 30/155] z_21cm=0.7776 (Δz=0.0001): Sign=-, T21_mean=0.00 mK | ETA: 370.4s
  [ 40/155] z_21cm=1.1669 (Δz=0.0004): Sign=-, T21_mean=0.00 mK | ETA: 337.6s
  [ 50/155] z_21cm=1.6414 (Δz=0.0011): Sign=-,

In [ ]:
# =============================================================================
# NOT FOR REPORTS
# PLOT: kSZ, kSZ², and 21cm Maps Side-by-Side at Selected Redshifts
# =============================================================================

print(f"\n=== PLOTTING kSZ vs kSZ² vs 21cm MAPS ===")

# Select a few representative redshifts to plot
# Get ionization fraction at each redshift
z_nodes_sorted = lightcone.node_redshifts[::-1]
x_e_nodes = 1.0 - lightcone.global_xH[::-1]

# Find redshifts closest to x_e = 0.2, 0.5, 0.9
target_xe = [0.2, 0.5, 0.9]
selected_z_plot = []

for xe_target in target_xe:
    idx = np.argmin(np.abs(x_e_nodes - xe_target))
    z_sel = z_nodes_sorted[idx]
    # Find closest node redshift from our results
    z_closest = min(cross_corr_results.keys(), key=lambda z: abs(z - z_sel))
    if abs(z_closest - z_sel) < 0.5:  # Reasonable match
        selected_z_plot.append(z_closest)

print(f"Plotting kSZ vs kSZ² vs 21cm for {len(selected_z_plot)} redshifts")

# Create figure: rows = redshifts, cols = [kSZ, kSZ², 21cm]
fig, axes = plt.subplots(len(selected_z_plot), 3, 
                         figsize=(16, 5*len(selected_z_plot)), 
                         constrained_layout=True)

if len(selected_z_plot) == 1:
    axes = axes.reshape(1, -1)

# Get lightcone redshift axis
lc_redshifts = np.asarray(lightcone.lightcone_redshifts, dtype=np.float64)

for row_idx, z_obs in enumerate(selected_z_plot):
    
    # Load kSZ map
    kSZ_map_file = f"{kSZ_maps_dir}/kSZ_map_z{z_obs:.6f}.npy"
    kSZ_map = np.load(kSZ_map_file)
    
    # Square it
    kSZ2_map = kSZ_map**2
    
    # Find closest lightcone slice to z_obs
    idx_closest = np.argmin(np.abs(lc_redshifts - z_obs))
    z_actual = lc_redshifts[idx_closest]
    
    # Extract 21cm brightness temperature slice
    T21_slice = np.asarray(lightcone.brightness_temp[:, :, idx_closest])
    
    # Get ionization fraction
    x_e = np.interp(z_obs, z_nodes_sorted, x_e_nodes)
    
    # =============================================================================
    # Left panel: kSZ map
    # =============================================================================
    
    ax_kSZ = axes[row_idx, 0]
    
    # Symmetric color scale for kSZ
    vmax_kSZ = np.percentile(np.abs(kSZ_map), 99)
    
    im_kSZ = ax_kSZ.imshow(kSZ_map.T,
                           cmap='seismic',
                           origin='lower',
                           extent=[0, box_size_Mpc, 0, box_size_Mpc],
                           aspect='equal',
                           vmin=-vmax_kSZ,
                           vmax=vmax_kSZ)
    
    # Colorbar
    cbar_kSZ = plt.colorbar(im_kSZ, ax=ax_kSZ, fraction=0.046, pad=0.04)
    cbar_kSZ.set_label('kSZ (dimensionless)', fontsize=12)
    
    # Labels
    ax_kSZ.set_xlabel('x [Mpc]', fontsize=14)
    ax_kSZ.set_ylabel('y [Mpc]', fontsize=14)
    ax_kSZ.set_title(f'kSZ Map\nz={z_obs:.2f}, $x_e$={x_e:.2f}', 
                     fontsize=14, fontweight='bold')
    
    # Stats
    rms_kSZ = np.sqrt(np.mean(kSZ_map**2))
    ax_kSZ.text(0.05, 0.95, 
               f'RMS={rms_kSZ:.2e}\nMean={kSZ_map.mean():.2e}',
               transform=ax_kSZ.transAxes, fontsize=11,
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # =============================================================================
    # Middle panel: kSZ² map
    # =============================================================================
    
    ax_kSZ2 = axes[row_idx, 1]
    
    # Use 'hot' colormap for squared map (all positive)
    vmax_kSZ2 = np.percentile(kSZ2_map, 99)
    
    im_kSZ2 = ax_kSZ2.imshow(kSZ2_map.T,
                             cmap='hot',
                             origin='lower',
                             extent=[0, box_size_Mpc, 0, box_size_Mpc],
                             aspect='equal',
                             vmin=0,
                             vmax=vmax_kSZ2)
    
    # Colorbar
    cbar_kSZ2 = plt.colorbar(im_kSZ2, ax=ax_kSZ2, fraction=0.046, pad=0.04)
    cbar_kSZ2.set_label('kSZ² (dimensionless)', fontsize=12)
    
    # Labels
    ax_kSZ2.set_xlabel('x [Mpc]', fontsize=14)
    ax_kSZ2.set_ylabel('y [Mpc]', fontsize=14)
    ax_kSZ2.set_title(f'kSZ² Map\nz={z_obs:.2f}, $x_e$={x_e:.2f}', 
                      fontsize=14, fontweight='bold')
    
    # Stats
    rms_kSZ2 = np.sqrt(np.mean(kSZ2_map**2))
    ax_kSZ2.text(0.05, 0.95, 
                f'RMS={rms_kSZ2:.2e}\nMean={kSZ2_map.mean():.2e}',
                transform=ax_kSZ2.transAxes, fontsize=11,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # =============================================================================
    # Right panel: 21cm brightness temperature map
    # =============================================================================
    
    ax_T21 = axes[row_idx, 2]
    
    # Use 'RdBu_r' or 'coolwarm' for 21cm (can be positive or negative)
    vmax_T21 = np.percentile(np.abs(T21_slice), 99)
    
    im_T21 = ax_T21.imshow(T21_slice.T,
                           cmap='RdBu_r',
                           origin='lower',
                           extent=[0, box_size_Mpc, 0, box_size_Mpc],
                           aspect='equal',
                           vmin=-vmax_T21,
                           vmax=vmax_T21)
    
    # Colorbar
    cbar_T21 = plt.colorbar(im_T21, ax=ax_T21, fraction=0.046, pad=0.04)
    cbar_T21.set_label('21cm Brightness Temp [mK]', fontsize=12)
    
    # Labels
    ax_T21.set_xlabel('x [Mpc]', fontsize=14)
    ax_T21.set_ylabel('y [Mpc]', fontsize=14)
    ax_T21.set_title(f'21cm Map\nz={z_actual:.2f}, $x_e$={x_e:.2f}', 
                     fontsize=14, fontweight='bold')
    
    # Stats
    rms_T21 = np.sqrt(np.mean(T21_slice**2))
    ax_T21.text(0.05, 0.95, 
               f'RMS={rms_T21:.2f} mK\nMean={T21_slice.mean():.2f} mK',
               transform=ax_T21.transAxes, fontsize=11,
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Overall title
fig.suptitle('kSZ, kSZ², and 21cm Maps at Different Reionization Epochs', 
            fontsize=20, fontweight='bold')

# Save
plot_name = "kSZ_kSZ2_21cm_maps_comparison"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

print("\n✓ kSZ vs kSZ² vs 21cm COMPARISON PLOTTING COMPLETE!")


=== PLOTTING kSZ vs kSZ² vs 21cm MAPS ===
Plotting kSZ vs kSZ² vs 21cm for 3 redshifts
✓ Saved: kSZ_kSZ2_21cm_maps_comparison

✓ kSZ vs kSZ² vs 21cm COMPARISON PLOTTING COMPLETE!


In [ ]:
# =============================================================================
# NOT FOR REPORTS
# PLOT 1: 2D FFT Maps (k-space) - kSZ² and 21cm Side-by-Side
# =============================================================================

print(f"\n=== PLOTTING 2D FFT MAPS IN k-SPACE ===")

# Select a few representative redshifts to plot
# Get ionization fraction at each redshift
z_nodes_sorted = lightcone.node_redshifts[::-1]
x_e_nodes = 1.0 - lightcone.global_xH[::-1]

# Find redshifts closest to x_e = 0.2, 0.5, 0.9
target_xe = [0.2, 0.5, 0.9]
selected_z_fft = []

for xe_target in target_xe:
    idx = np.argmin(np.abs(x_e_nodes - xe_target))
    z_sel = z_nodes_sorted[idx]
    # Find closest node redshift from our results
    z_closest = min(cross_corr_results.keys(), key=lambda z: abs(z - z_sel))
    if abs(z_closest - z_sel) < 0.5:  # Reasonable match
        selected_z_fft.append(z_closest)

print(f"Plotting 2D FFT for {len(selected_z_fft)} redshifts")

# Create figure: rows = redshifts, cols = [kSZ² FFT, 21cm FFT]
fig, axes = plt.subplots(len(selected_z_fft), 2, 
                         figsize=(14, 6*len(selected_z_fft)), 
                         constrained_layout=True)

if len(selected_z_fft) == 1:
    axes = axes.reshape(1, -1)

# Get lightcone redshift axis
lc_redshifts = np.asarray(lightcone.lightcone_redshifts, dtype=np.float64)

for row_idx, z_obs in enumerate(selected_z_fft):
    
    # Recompute FFTs for this redshift
    # Load kSZ map
    kSZ_map_file = f"{kSZ_maps_dir}/kSZ_map_z{z_obs:.6f}.npy"
    kSZ_map = np.load(kSZ_map_file)
    kSZ2_map = kSZ_map**2
    kSZ2_map_centered = kSZ2_map - np.mean(kSZ2_map)
    
    # Get 21cm slice
    idx_closest = np.argmin(np.abs(lc_redshifts - z_obs))
    T21_slice = np.asarray(lightcone.brightness_temp[:, :, idx_closest])
    T21_slice_centered = T21_slice - np.mean(T21_slice)
    
    # Compute FFTs
    fft_kSZ2 = np.fft.fft2(kSZ2_map_centered)
    fft_kSZ2_shifted = np.fft.fftshift(fft_kSZ2)
    
    fft_T21 = np.fft.fft2(T21_slice_centered)
    fft_T21_shifted = np.fft.fftshift(fft_T21)
    
    # Get ionization fraction
    x_e = np.interp(z_obs, z_nodes_sorted, x_e_nodes)
    
    # k-space extent
    k_max = kgrid.max()
    
    # =============================================================================
    # Left panel: kSZ² FFT
    # =============================================================================
    
    ax_fft_kSZ2 = axes[row_idx, 0]
    
    # Plot log10 of power
    power_kSZ2 = np.abs(fft_kSZ2_shifted)**2
    power_kSZ2_log = np.log10(power_kSZ2 + 1e-20)  # Add small value to avoid log(0)
    
    im_fft_kSZ2 = ax_fft_kSZ2.imshow(power_kSZ2_log.T,
                                      cmap='viridis',
                                      origin='lower',
                                      extent=[-k_max, k_max, -k_max, k_max],
                                      aspect='equal')
    
    # Colorbar
    cbar_fft_kSZ2 = plt.colorbar(im_fft_kSZ2, ax=ax_fft_kSZ2, fraction=0.046, pad=0.04)
    cbar_fft_kSZ2.set_label(r'log$_{10}$(Power)', fontsize=12)
    
    # Labels
    ax_fft_kSZ2.set_xlabel(r'$k_x$ [Mpc$^{-1}$]', fontsize=14)
    ax_fft_kSZ2.set_ylabel(r'$k_y$ [Mpc$^{-1}$]', fontsize=14)
    ax_fft_kSZ2.set_title(f'kSZ² Power (k-space)\nz={z_obs:.2f}, $x_e$={x_e:.2f}', 
                          fontsize=14, fontweight='bold')
    
    # Add circle at k = 0.1 Mpc^-1 for reference
    circle = plt.Circle((0, 0), 0.1, color='white', fill=False, linestyle='--', linewidth=1.5)
    ax_fft_kSZ2.add_patch(circle)
    
    # =============================================================================
    # Right panel: 21cm FFT
    # =============================================================================
    
    ax_fft_T21 = axes[row_idx, 1]
    
    # Plot log10 of power
    power_T21 = np.abs(fft_T21_shifted)**2
    power_T21_log = np.log10(power_T21 + 1e-20)
    
    im_fft_T21 = ax_fft_T21.imshow(power_T21_log.T,
                                    cmap='viridis',
                                    origin='lower',
                                    extent=[-k_max, k_max, -k_max, k_max],
                                    aspect='equal')
    
    # Colorbar
    cbar_fft_T21 = plt.colorbar(im_fft_T21, ax=ax_fft_T21, fraction=0.046, pad=0.04)
    cbar_fft_T21.set_label(r'log$_{10}$(Power)', fontsize=12)
    
    # Labels
    ax_fft_T21.set_xlabel(r'$k_x$ [Mpc$^{-1}$]', fontsize=14)
    ax_fft_T21.set_ylabel(r'$k_y$ [Mpc$^{-1}$]', fontsize=14)
    ax_fft_T21.set_title(f'21cm Power (k-space)\nz={z_obs:.2f}, $x_e$={x_e:.2f}', 
                         fontsize=14, fontweight='bold')
    
    # Add circle at k = 0.1 Mpc^-1 for reference
    circle = plt.Circle((0, 0), 0.1, color='white', fill=False, linestyle='--', linewidth=1.5)
    ax_fft_T21.add_patch(circle)

# Overall title
fig.suptitle('2D Power Spectra in k-space', 
            fontsize=20, fontweight='bold')

# Save
plot_name = "2D_FFT_kspace_kSZ2_21cm"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT 2: Cross-Power and Auto-Power Spectra vs k
# =============================================================================

print(f"\n=== PLOTTING POWER SPECTRA vs k ===")

# Plot for the same selected redshifts
fig, axes = plt.subplots(len(selected_z_fft), 1, 
                         figsize=(10, 6*len(selected_z_fft)), 
                         constrained_layout=True)

if len(selected_z_fft) == 1:
    axes = [axes]

for row_idx, z_obs in enumerate(selected_z_fft):
    
    if z_obs not in cross_corr_results:
        continue
    
    results = cross_corr_results[z_obs]
    k_centers = results['k_centers']
    C_cross = results['C_cross_1d']
    P_kSZ2 = results['P_kSZ2_1d']
    P_T21 = results['P_T21_1d']
    
    # Get ionization fraction
    x_e = np.interp(z_obs, z_nodes_sorted, x_e_nodes)
    
    ax = axes[row_idx]
    
    # Filter valid points
    valid_cross = ~np.isnan(C_cross) & np.isfinite(C_cross)
    valid_kSZ2 = ~np.isnan(P_kSZ2) & (P_kSZ2 > 0)
    valid_T21 = ~np.isnan(P_T21) & (P_T21 > 0)
    
    # Plot auto-power spectra
    ax.loglog(k_centers[valid_kSZ2], P_kSZ2[valid_kSZ2], 
             'o-', color='red', linewidth=2, markersize=4,
             label='kSZ² Auto-Power', alpha=0.8)
    
    ax.loglog(k_centers[valid_T21], P_T21[valid_T21], 
             's-', color='blue', linewidth=2, markersize=4,
             label='21cm Auto-Power', alpha=0.8)
    
    # Plot cross-power (can be negative, so plot absolute value)
    # Use different markers for positive/negative
    positive_mask = valid_cross & (C_cross > 0)
    negative_mask = valid_cross & (C_cross < 0)
    
    if np.any(positive_mask):
        ax.loglog(k_centers[positive_mask], np.abs(C_cross[positive_mask]), 
                 '^-', color='green', linewidth=2.5, markersize=6,
                 label='|Cross-Power| (positive)', alpha=0.9)
    
    if np.any(negative_mask):
        ax.loglog(k_centers[negative_mask], np.abs(C_cross[negative_mask]), 
                 'v--', color='purple', linewidth=2.5, markersize=6,
                 label='|Cross-Power| (negative)', alpha=0.9)
    
    ax.set_xlabel(r'$k$ [Mpc$^{-1}$]', fontsize=16)
    ax.set_ylabel(r'Power [Mpc$^2$]', fontsize=16)
    ax.set_title(f'Power Spectra: z={z_obs:.2f}, $x_e$={x_e:.2f}', 
                fontsize=16, fontweight='bold')
    ax.legend(fontsize=12, loc='best')
    ax.grid(True, alpha=0.3)
    
    # Add text showing sign of cross-power
    if np.any(valid_cross):
        mean_sign = "positive" if np.mean(C_cross[valid_cross]) > 0 else "negative"
        ax.text(0.05, 0.95, f'Cross-power: {mean_sign}',
               transform=ax.transAxes, fontsize=12, fontweight='bold',
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Overall title
fig.suptitle('Power Spectra vs k (kSZ², 21cm, and Cross-Power)', 
            fontsize=18, fontweight='bold')

# Save
plot_name = "power_spectra_vs_k_comparison"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT 3: Cross-Power Sign Evolution vs k at Different Redshifts
# =============================================================================

print(f"\n=== PLOTTING CROSS-POWER SIGN EVOLUTION ===")

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

# Select more redshifts for this plot
z_sample = sorted(cross_corr_results.keys())[::15]  # Every 15th redshift

cmap = mpl.cm.rainbow
norm = mpl.colors.Normalize(vmin=min(z_sample), vmax=max(z_sample))

for z_obs in z_sample:
    results = cross_corr_results[z_obs]
    k_centers = results['k_centers']
    C_cross = results['C_cross_1d']
    
    valid = ~np.isnan(C_cross) & np.isfinite(C_cross)
    
    if np.sum(valid) > 5:
        color = cmap(norm(z_obs))
        
        # Plot with sign preserved (use symlog or just regular plot)
        ax.plot(k_centers[valid], C_cross[valid], 
               color=color, linewidth=2, alpha=0.7,
               marker='o', markersize=3)

ax.set_xlabel(r'$k$ [Mpc$^{-1}$]', fontsize=18)
ax.set_ylabel(r'Cross-Power [Mpc$^2$]', fontsize=18)
ax.set_xscale('log')
ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.grid(True, alpha=0.3)

# Add colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r'Redshift $z$', fontsize=16)

ax.set_title('kSZ²-21cm Cross-Power vs k (Sign Evolution)', 
            fontsize=18, fontweight='bold')

# Save
plot_name = "cross_power_vs_k_sign_evolution"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

print("\n✓ ALL DIAGNOSTIC PLOTTING COMPLETE!")


=== PLOTTING 2D FFT MAPS IN k-SPACE ===
Plotting 2D FFT for 3 redshifts
✓ Saved: 2D_FFT_kspace_kSZ2_21cm

=== PLOTTING POWER SPECTRA vs k ===
✓ Saved: power_spectra_vs_k_comparison

=== PLOTTING CROSS-POWER SIGN EVOLUTION ===
✓ Saved: cross_power_vs_k_sign_evolution

✓ ALL DIAGNOSTIC PLOTTING COMPLETE!


In [ ]:
# =============================================================================
# CELL 8: Visualize kSZ²-21cm Cross-Correlation Power Spectra (Random Seed)
# Convert to ℓ-space and create plots for one randomly selected realisation
# =============================================================================

print("\n" + "="*70)
print("VISUALIZING kSZ²-21cm CROSS-CORRELATION POWER SPECTRA")
print("="*70)

# Create subdirectory for final visualization plots
plot_dir_final = f"{plot_dir}/plot_final_cell"
if not os.path.exists(plot_dir_final):
    os.makedirs(plot_dir_final)
plot_dir_save = plot_dir_final

# ==========================================================================
# Load cross_corr_results_all from cache if not in memory
# ==========================================================================
if 'cross_corr_results_all' not in dir() or len(cross_corr_results_all) == 0:
    print("cross_corr_results_all not in memory → loading from Cell 7 cache")
    cross_corr_results_all = {}
    for seed in RANDOM_SEEDS:
        cc_cache = f"{cache_dir}/seed_{seed}/cross_corr_seed{seed}.npy"
        if os.path.exists(cc_cache):
            cross_corr_results_all[seed] = np.load(
                cc_cache, allow_pickle=True).item()
            print(f"  ✓ Loaded seed {seed} "
                  f"({len(cross_corr_results_all[seed])} redshifts)")
        else:
            print(f"  ✗ No cache found for seed {seed}")
    print(f"  Loaded {len(cross_corr_results_all)}/{N_SEEDS} seeds")
else:
    print(f"cross_corr_results_all already in memory "
          f"({len(cross_corr_results_all)} seeds)")

if len(cross_corr_results_all) > 0:

    # Pick one random seed to plot
    seed_to_plot = int(np.random.choice(list(cross_corr_results_all.keys())))
    print(f"\nRandomly selected seed for plots: {seed_to_plot}")

    cross_corr_results = cross_corr_results_all[seed_to_plot]
    lc                 = lightcones[seed_to_plot]

    # ==========================================================================
    # Convert k-space to ℓ-space
    # ==========================================================================

    print(f"\n=== CONVERTING TO ℓ-SPACE WITH ERROR PROPAGATION ===")

    T_CMB_0_K = 2.725
    from astropy.cosmology import FlatLambdaCDM
    cosmo = FlatLambdaCDM(H0=67.77, Om0=0.3086)

    cross_corr_ell_results = {}

    for z_obs in sorted(cross_corr_results.keys()):
        results          = cross_corr_results[z_obs]
        D_A_Mpc          = float(cosmo.angular_diameter_distance(z_obs).value)
        chi_comoving_Mpc = float(cosmo.comoving_distance(z_obs).value)
        T_CMB_z_uK       = T_CMB_0_K * 1e6

        k_centers  = results['k_centers']
        ell_from_k = k_centers * chi_comoving_Mpc / 0.67

        C_cross_ell            = results['C_cross_1d']            * 0.67**2 / D_A_Mpc**2
        C_cross_ell_err_sample = results['C_cross_1d_err_sample'] * 0.67**2 / D_A_Mpc**2
        C_cross_ell_err_cosmic = results['C_cross_1d_err_cosmic'] * 0.67**2 / D_A_Mpc**2
        C_cross_ell_err_total  = results['C_cross_1d_err_total']  * 0.67**2 / D_A_Mpc**2

        D_cross_ell            = ell_from_k * (ell_from_k + 1) * C_cross_ell            / (2 * np.pi)
        D_cross_ell_err_sample = ell_from_k * (ell_from_k + 1) * C_cross_ell_err_sample / (2 * np.pi)
        D_cross_ell_err_cosmic = ell_from_k * (ell_from_k + 1) * C_cross_ell_err_cosmic / (2 * np.pi)
        D_cross_ell_err_total  = ell_from_k * (ell_from_k + 1) * C_cross_ell_err_total  / (2 * np.pi)

        D_cross_ell_uK_mK            = D_cross_ell            * T_CMB_z_uK**2
        D_cross_ell_uK_mK_err_sample = D_cross_ell_err_sample * T_CMB_z_uK**2
        D_cross_ell_uK_mK_err_cosmic = D_cross_ell_err_cosmic * T_CMB_z_uK**2
        D_cross_ell_uK_mK_err_total  = D_cross_ell_err_total  * T_CMB_z_uK**2

        P_kSZ2_ell = results['P_kSZ2_1d'] * 0.67**2 / D_A_Mpc**2
        P_T21_ell  = results['P_T21_1d']  * 0.67**2 / D_A_Mpc**2
        with np.errstate(divide='ignore', invalid='ignore'):
            r_cross = C_cross_ell / np.sqrt(P_kSZ2_ell * P_T21_ell)

        cross_corr_ell_results[z_obs] = {
            'ell_from_k'                   : ell_from_k,
            'D_cross_ell_uK_mK'            : D_cross_ell_uK_mK,
            'D_cross_ell_uK_mK_err_sample' : D_cross_ell_uK_mK_err_sample,
            'D_cross_ell_uK_mK_err_cosmic' : D_cross_ell_uK_mK_err_cosmic,
            'D_cross_ell_uK_mK_err_total'  : D_cross_ell_uK_mK_err_total,
            'D_cross_ell_dimensionless'     : D_cross_ell,
            'r_cross'                       : r_cross,
            'D_A_Mpc'                       : D_A_Mpc,
            'T_CMB_z_uK'                    : T_CMB_z_uK
        }

    print(f"Converted {len(cross_corr_ell_results)} redshifts to ℓ-space")

    slabel   = f"seed{seed_to_plot}"
    z_values = np.array(sorted(cross_corr_ell_results.keys()))
    cmap     = mpl.cm.rainbow
    norm     = mpl.colors.Normalize(vmin=z_values.min(), vmax=z_values.max())

    # ==========================================================================
    # PLOT 1: Rainbow D_ℓ vs ℓ
    # ==========================================================================

    print(f"\n=== PLOT 1: Rainbow D_ℓ vs ℓ ===")

    fig, ax = plt.subplots(1, 1, figsize=(12, 8), constrained_layout=True)

    for z_obs in z_values[::2]:
        results   = cross_corr_ell_results[z_obs]
        ell       = results['ell_from_k']
        D_ell     = results['D_cross_ell_uK_mK']
        D_ell_err = results['D_cross_ell_uK_mK_err_total']
        valid     = ~np.isnan(D_ell) & np.isfinite(D_ell) & (ell > 10) & ~np.isnan(D_ell_err)
        if np.sum(valid) > 5:
            color = cmap(norm(z_obs))
            ax.plot(ell[valid], D_ell[valid], color=color, lw=1.5, alpha=0.8)
            ax.fill_between(ell[valid],
                            D_ell[valid] - D_ell_err[valid],
                            D_ell[valid] + D_ell_err[valid],
                            color=color, alpha=0.15)

    ax.set_xlabel(r'Multipole $\ell$', fontsize=14)
    ax.set_ylabel(r'$D_\ell$ [kSZ$^2$-21cm] (μK$^2$·mK)', fontsize=14)
    ax.set_xscale('log')
    ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    plt.colorbar(sm, ax=ax, pad=0.02).set_label(r'Redshift $z$', fontsize=12)
    ax.text(0.02, 0.02, f'seed={seed_to_plot}', transform=ax.transAxes,
            fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plot_name = f"kSZ2_21cm_cross_Dl_vs_ell_rainbow_{slabel}"
    fig.savefig(f"{plot_dir_save}/{plot_name}.pdf", bbox_inches='tight')
    ax.set_title(r'kSZ$^2$-21cm Cross-Power $D_\ell$ vs Redshift',
                 fontsize=16, fontweight='bold')
    fig.savefig(f"{plot_dir_save}/{plot_name}.png", dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_name}")
    plt.close(fig)

    # ==========================================================================
    # PLOT 2: Correlation Coefficient r vs ℓ
    # ==========================================================================

    print(f"\n=== PLOT 2: Correlation Coefficient r vs ℓ ===")

    fig, ax = plt.subplots(1, 1, figsize=(12, 8), constrained_layout=True)

    for z_obs in z_values:
        results = cross_corr_ell_results[z_obs]
        ell     = results['ell_from_k']
        r       = results['r_cross']
        valid   = ~np.isnan(r) & np.isfinite(r) & (ell > 10) & (np.abs(r) < 1.5)
        if np.sum(valid) > 5:
            ax.plot(ell[valid], r[valid], color=cmap(norm(z_obs)), lw=1.5, alpha=0.7)

    ax.set_xlabel(r'Multipole $\ell$', fontsize=14)
    ax.set_ylabel(r'Correlation Coefficient $r$', fontsize=14)
    ax.set_xscale('log')
    ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
    ax.set_ylim(-1.2, 1.2)
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    plt.colorbar(sm, ax=ax, pad=0.02).set_label(r'Redshift $z$', fontsize=12)
    ax.text(0.02, 0.02, f'seed={seed_to_plot}', transform=ax.transAxes,
            fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plot_name = f"kSZ2_21cm_cross_r_vs_ell_rainbow_{slabel}"
    fig.savefig(f"{plot_dir_save}/{plot_name}.pdf", bbox_inches='tight')
    ax.set_title(r'kSZ$^2$-21cm Correlation Coefficient vs Redshift',
                 fontsize=16, fontweight='bold')
    fig.savefig(f"{plot_dir_save}/{plot_name}.png", dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_name}")
    plt.close(fig)

    # ==========================================================================
    # PLOT 3: Selected x_e with error bars
    # ==========================================================================

    print(f"\n=== PLOT 3: Selected ionization fractions ===")

    z_nodes_sorted = lc.node_redshifts[::-1]
    x_e_nodes      = 1.0 - lc.global_xH[::-1]
    target_xe      = [0.2, 0.5, 0.9]
    selected_z     = [z_nodes_sorted[np.argmin(np.abs(x_e_nodes - xe))] for xe in target_xe]
    selected_xe    = [x_e_nodes[np.argmin(np.abs(x_e_nodes - xe))]      for xe in target_xe]

    print("  Selected redshifts:")
    for z, xe in zip(selected_z, selected_xe):
        print(f"    z={z:.2f}, x_e={xe:.3f}")

    colors_selected = ['blue', 'green', 'red']
    fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

    for i, (z_obs, xe) in enumerate(zip(selected_z, selected_xe)):
        z_closest = min(cross_corr_ell_results.keys(), key=lambda z: abs(z - z_obs))
        if abs(z_closest - z_obs) > 0.5:
            continue
        results          = cross_corr_ell_results[z_closest]
        ell              = results['ell_from_k']
        D_ell            = results['D_cross_ell_uK_mK']
        D_ell_err_total  = results['D_cross_ell_uK_mK_err_total']
        D_ell_err_sample = results['D_cross_ell_uK_mK_err_sample']
        valid = (~np.isnan(D_ell) & np.isfinite(D_ell)
                 & (ell > 10) & ~np.isnan(D_ell_err_total))
        if np.sum(valid) > 5:
            ax.errorbar(ell[valid], D_ell[valid],
                        yerr=D_ell_err_total[valid],
                        color=colors_selected[i], lw=2.5, alpha=0.8,
                        marker='o', markersize=5, capsize=3, capthick=1.5,
                        label=f'z={z_closest:.1f} ($x_e$={xe:.2f})',
                        errorevery=3)
            ax.fill_between(ell[valid],
                            D_ell[valid] - D_ell_err_sample[valid],
                            D_ell[valid] + D_ell_err_sample[valid],
                            color=colors_selected[i], alpha=0.15)

    ax.set_xlabel(r'Multipole $\ell$', fontsize=14)
    ax.set_ylabel(r'$D_\ell$ [kSZ$^2$-21cm] (μK$^2$·mK)', fontsize=14)
    ax.set_xscale('log')
    ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
    ax.legend(fontsize=14, loc='best', framealpha=0.9)
    ax.grid(True, alpha=0.3, ls='--')
    ax.text(0.02, 0.02, f'seed={seed_to_plot}', transform=ax.transAxes,
            fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plot_name = f"kSZ2_21cm_cross_Dl_selected_xe_{slabel}"
    fig.savefig(f"{plot_dir_save}/{plot_name}.pdf", bbox_inches='tight')
    ax.set_title(r'kSZ$^2$-21cm Cross-Power at Key Ionization Fractions',
                 fontsize=16, fontweight='bold')
    fig.savefig(f"{plot_dir_save}/{plot_name}.png", dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_name}")
    plt.close(fig)

    # ==========================================================================
    # PLOT 4: D_ℓ vs z at fixed ℓ
    # ==========================================================================

    print(f"\n=== PLOT 4: D_ℓ evolution at fixed ℓ ===")

    ell_targets = [500, 1000, 3000]
    colors_ell  = ['darkblue', 'darkgreen', 'darkred']

    fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

    for i, ell_target in enumerate(ell_targets):
        z_plot, D_plot, D_err_plot = [], [], []
        for z_obs in sorted(cross_corr_ell_results.keys()):
            results = cross_corr_ell_results[z_obs]
            ell     = results['ell_from_k']
            D_ell   = results['D_cross_ell_uK_mK']
            D_err   = results['D_cross_ell_uK_mK_err_total']
            idx     = np.argmin(np.abs(ell - ell_target))
            if np.isfinite(D_ell[idx]) and np.isfinite(D_err[idx]):
                z_plot.append(z_obs)
                D_plot.append(D_ell[idx])
                D_err_plot.append(D_err[idx])

        if len(z_plot) > 0:
            z_plot     = np.array(z_plot)
            D_plot     = np.array(D_plot)
            D_err_plot = np.array(D_err_plot)
            ax.errorbar(z_plot, D_plot, yerr=D_err_plot,
                        color=colors_ell[i], lw=2.5, alpha=0.8,
                        marker='o', markersize=5, capsize=4, capthick=1.5,
                        label=f'$\\ell$={ell_target}', errorevery=2)
            ax.fill_between(z_plot, D_plot - D_err_plot, D_plot + D_err_plot,
                            color=colors_ell[i], alpha=0.15)

    ax.set_xlabel(r'Redshift $z$', fontsize=14)
    ax.set_ylabel(r'$D_\ell$ [kSZ$^2$-21cm] (μK$^2$·mK)', fontsize=14)
    ax.set_yscale('symlog', linthresh=1e-2)
    ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
    ax.legend(fontsize=14, loc='best', framealpha=0.9)
    ax.invert_xaxis()
    ax.grid(True, alpha=0.3, ls='--')
    ax.text(0.05, 0.95, f'seed={seed_to_plot}', transform=ax.transAxes,
            fontsize=11, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plot_name = f"kSZ2_21cm_cross_Dl_vs_z_fixed_ell_{slabel}"
    fig.savefig(f"{plot_dir_save}/{plot_name}.pdf", bbox_inches='tight')
    ax.set_title(r'kSZ$^2$-21cm Cross-Power Evolution at Fixed $\ell$',
                 fontsize=16, fontweight='bold')
    fig.savefig(f"{plot_dir_save}/{plot_name}.png", dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_name}")
    plt.close(fig)

    # ==========================================================================
    # PLOT 5: Error Budget
    # ==========================================================================

    print(f"\n=== PLOT 5: Error budget ===")

    z_example        = min(cross_corr_ell_results.keys(),
                           key=lambda z: abs(z - selected_z[1]))
    results          = cross_corr_ell_results[z_example]
    ell              = results['ell_from_k']
    D_ell            = results['D_cross_ell_uK_mK']
    D_ell_err_sample = results['D_cross_ell_uK_mK_err_sample']
    D_ell_err_cosmic = results['D_cross_ell_uK_mK_err_cosmic']
    D_ell_err_total  = results['D_cross_ell_uK_mK_err_total']
    valid = ~np.isnan(D_ell) & (ell > 10) & (D_ell != 0)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10),
                                   constrained_layout=True, sharex=True)

    ax1.errorbar(ell[valid], D_ell[valid], yerr=D_ell_err_total[valid],
                 fmt='o-', color='darkblue', lw=2, markersize=4,
                 capsize=3, alpha=0.8, label='Cross-power ± total error')
    ax1.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
    ax1.set_ylabel(r'$D_\ell$ [μK$^2$·mK]', fontsize=12)
    ax1.set_xscale('log')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax1.set_title(f'Error Budget (z={z_example:.2f}, seed={seed_to_plot})',
                  fontsize=14, fontweight='bold')

    frac_sample = D_ell_err_sample[valid] / np.abs(D_ell[valid]) * 100
    frac_cosmic = D_ell_err_cosmic[valid] / np.abs(D_ell[valid]) * 100
    frac_total  = D_ell_err_total[valid]  / np.abs(D_ell[valid]) * 100

    ax2.plot(ell[valid], frac_sample, 'o-', color='blue',  lw=2, markersize=4,
             alpha=0.7, label='Sample variance')
    ax2.plot(ell[valid], frac_cosmic, 's-', color='red',   lw=2, markersize=4,
             alpha=0.7, label='Cosmic variance')
    ax2.plot(ell[valid], frac_total,  '^-', color='black', lw=2.5, markersize=5,
             alpha=0.8, label='Total')
    ax2.set_xlabel(r'Multipole $\ell$', fontsize=12)
    ax2.set_ylabel('Fractional Error (%)', fontsize=12)
    ax2.set_xscale('log')
    ax2.set_yscale('log')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    plot_name = f"kSZ2_21cm_cross_error_budget_{slabel}"
    fig.savefig(f"{plot_dir_save}/{plot_name}.pdf", bbox_inches='tight')
    fig.savefig(f"{plot_dir_save}/{plot_name}.png", dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_name}")
    plt.close(fig)

    print(f"\n✓ ALL PLOTS COMPLETE (seed={seed_to_plot})")

else:
    print("\n✗ Skipping — no cross_corr_results_all available")

print("\n" + "="*70)


VISUALIZING kSZ²-21cm CROSS-CORRELATION POWER SPECTRA WITH ERROR BARS
Created final plots directory: 18March2026_kSZ2_21cm/plots/plot_final_cell

=== CONVERTING TO ℓ-SPACE WITH ERROR PROPAGATION ===
Converted 155 redshifts to ℓ-space

=== GENERATING RAINBOW PLOT WITH ERROR BANDS ===
✓ Saved: kSZ2_21cm_cross_Dl_vs_ell_rainbow_with_errors

=== GENERATING CORRELATION COEFFICIENT PLOT ===


/tmp/ipykernel_131538/3777405291.py:155: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


✓ Saved: kSZ2_21cm_cross_r_vs_ell_rainbow

=== GENERATING SELECTED REDSHIFTS PLOT WITH ERROR BARS ===

Selected redshifts for labeled plot:
  z=9.78, x_e=0.205
  z=8.02, x_e=0.510
  z=6.40, x_e=0.903
✓ Saved: kSZ2_21cm_cross_Dl_selected_xe_with_errors

=== GENERATING EVOLUTION PLOT WITH ERROR BARS ===
✓ Saved: kSZ2_21cm_cross_Dl_vs_z_fixed_ell_with_errors

=== GENERATING ERROR BUDGET PLOT ===
✓ Saved: kSZ2_21cm_cross_error_budget

✓ ALL VISUALIZATION WITH ERROR BARS COMPLETE!



In [12]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import Planck18 as cosmo

# Redshift range
z = np.linspace(0.1, 20, 300)

# Distances
chi_comoving = cosmo.comoving_distance(z).value          # Mpc
D_A = cosmo.angular_diameter_distance(z).value           # Mpc

# Ratio
ratio = chi_comoving/ D_A

# Plot
fig, axs = plt.subplots(2, 1, figsize=(8, 10), sharex=True)

# Top panel: distances
axs[0].plot(z, chi_comoving, lw=2, label=r'Comoving distance $\chi(z)$')
axs[0].plot(z, D_A, lw=2, label=r'Angular diameter distance $D_A(z)$')
axs[0].set_ylabel('Distance [Mpc]')
axs[0].legend()
axs[0].grid(alpha=0.3)

# Bottom panel: ratio
axs[1].plot(z, ratio, lw=2, color='black')
axs[1].set_xlabel('Redshift $z$')
axs[1].set_ylabel(r'$\chi(z) / D_A(z)$')
axs[1].grid(alpha=0.3)

# Save
fig.savefig("distance_comoving_DA_and_ratio.pdf", dpi=300, bbox_inches="tight")
fig.savefig("distance_comoving_DA_and_ratio.png", dpi=300, bbox_inches="tight")

plt.show()

/tmp/ipykernel_2164853/465691316.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# =============================================================================
# CELL 8b: Visualize kSZ²-21cm Cross-Correlation with Redshift Binning
# Averages over ALL seeds — smooths noisy evolution curves
# =============================================================================

print("\n" + "="*70)
print("VISUALIZING kSZ²-21cm CROSS-CORRELATION (BINNED, ALL SEEDS)")
print("="*70)

# Create subdirectory for final visualization plots
plot_dir_final = f"{plot_dir}/plot_final_cell"
if not os.path.exists(plot_dir_final):
    os.makedirs(plot_dir_final)
    print(f"Created final plots directory: {plot_dir_final}")
else:
    print(f"Final plots directory exists: {plot_dir_final}")
plot_dir_save = plot_dir_final

# ==========================================================================
# Load cross_corr_results_all from cache if not in memory
# ==========================================================================
if 'cross_corr_results_all' not in dir() or len(cross_corr_results_all) == 0:
    print("cross_corr_results_all not in memory → loading from Cell 7 cache")
    cross_corr_results_all = {}
    for seed in RANDOM_SEEDS:
        cc_cache = f"{cache_dir}/seed_{seed}/cross_corr_seed{seed}.npy"
        if os.path.exists(cc_cache):
            cross_corr_results_all[seed] = np.load(
                cc_cache, allow_pickle=True).item()
            print(f"  ✓ Loaded seed {seed} "
                  f"({len(cross_corr_results_all[seed])} redshifts)")
        else:
            print(f"  ✗ No cache found for seed {seed}")
    print(f"  Loaded {len(cross_corr_results_all)}/{N_SEEDS} seeds")
else:
    print(f"cross_corr_results_all already in memory "
          f"({len(cross_corr_results_all)} seeds)")

if len(cross_corr_results_all) > 0:

    # ==========================================================================
    # Convert k → ℓ for ALL seeds
    # ==========================================================================

    print(f"\n=== CONVERTING TO ℓ-SPACE FOR ALL SEEDS ===")

    T_CMB_0_K = 2.725
    from astropy.cosmology import FlatLambdaCDM
    cosmo = FlatLambdaCDM(H0=67.77, Om0=0.3086)

    # {seed: {z_obs: ell_results_dict}}
    cross_corr_ell_all = {}

    for seed, ccr in cross_corr_results_all.items():
        cross_corr_ell_results = {}

        for z_obs in sorted(ccr.keys()):
            results          = ccr[z_obs]
            D_A_Mpc          = float(cosmo.angular_diameter_distance(z_obs).value)
            chi_comoving_Mpc = float(cosmo.comoving_distance(z_obs).value)
            T_CMB_z_uK       = T_CMB_0_K * 1e6

            k_centers  = results['k_centers']
            ell_from_k = k_centers * chi_comoving_Mpc / 0.67

            C_cross_ell            = results['C_cross_1d']            * 0.67**2 / D_A_Mpc**2
            C_cross_ell_err_sample = results['C_cross_1d_err_sample'] * 0.67**2 / D_A_Mpc**2
            C_cross_ell_err_cosmic = results['C_cross_1d_err_cosmic'] * 0.67**2 / D_A_Mpc**2
            C_cross_ell_err_total  = results['C_cross_1d_err_total']  * 0.67**2 / D_A_Mpc**2

            D_cross_ell            = ell_from_k * (ell_from_k+1) * C_cross_ell            / (2*np.pi)
            D_cross_ell_err_sample = ell_from_k * (ell_from_k+1) * C_cross_ell_err_sample / (2*np.pi)
            D_cross_ell_err_cosmic = ell_from_k * (ell_from_k+1) * C_cross_ell_err_cosmic / (2*np.pi)
            D_cross_ell_err_total  = ell_from_k * (ell_from_k+1) * C_cross_ell_err_total  / (2*np.pi)

            D_cross_ell_uK_mK            = D_cross_ell            * T_CMB_z_uK**2
            D_cross_ell_uK_mK_err_sample = D_cross_ell_err_sample * T_CMB_z_uK**2
            D_cross_ell_uK_mK_err_cosmic = D_cross_ell_err_cosmic * T_CMB_z_uK**2
            D_cross_ell_uK_mK_err_total  = D_cross_ell_err_total  * T_CMB_z_uK**2

            P_kSZ2_ell = results['P_kSZ2_1d'] * 0.67**2 / D_A_Mpc**2
            P_T21_ell  = results['P_T21_1d']  * 0.67**2 / D_A_Mpc**2
            with np.errstate(divide='ignore', invalid='ignore'):
                r_cross = C_cross_ell / np.sqrt(P_kSZ2_ell * P_T21_ell)

            cross_corr_ell_results[z_obs] = {
                'ell_from_k'                   : ell_from_k,
                'D_cross_ell_uK_mK'            : D_cross_ell_uK_mK,
                'D_cross_ell_uK_mK_err_sample' : D_cross_ell_uK_mK_err_sample,
                'D_cross_ell_uK_mK_err_cosmic' : D_cross_ell_uK_mK_err_cosmic,
                'D_cross_ell_uK_mK_err_total'  : D_cross_ell_uK_mK_err_total,
                'D_cross_ell_dimensionless'     : D_cross_ell,
                'r_cross'                       : r_cross,
                'D_A_Mpc'                       : D_A_Mpc,
                'T_CMB_z_uK'                    : T_CMB_z_uK
            }

        cross_corr_ell_all[seed] = cross_corr_ell_results

    print(f"Converted {len(cross_corr_ell_all)} seeds to ℓ-space")

    # Use first seed to get z list and ell grid (same for all seeds)
    ref_seed    = list(cross_corr_ell_all.keys())[0]
    ref_lc      = lightcones[ref_seed]
    all_z_nodes = sorted(cross_corr_ell_all[ref_seed].keys())

    # Redshift bins
    z_bins        = np.linspace(5, 20, 31)   # 30 bins, Δz ≈ 0.5
    z_bin_centers = 0.5 * (z_bins[:-1] + z_bins[1:])
    ell_targets   = [500, 1000, 3000]
    colors_ell    = ['darkblue', 'darkgreen', 'darkred']

    print(f"Redshift binning: {len(z_bins)-1} bins, "
          f"Δz ≈ {np.diff(z_bins).mean():.2f}")

    # ==========================================================================
    # PLOT 1: D_ℓ vs z at fixed ℓ — binned AND averaged across seeds
    # ==========================================================================

    print(f"\n=== PLOT 1: D_ℓ Evolution (Binned + Seed-Averaged) ===")

    fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

    for i, ell_target in enumerate(ell_targets):
        print(f"  Processing ℓ = {ell_target}...")

        D_binned   = []
        D_err_binned = []
        z_used     = []

        for j in range(len(z_bins) - 1):
            z_low, z_high = z_bins[j], z_bins[j+1]
            D_vals, D_errs = [], []

            # Collect from ALL seeds in this z bin
            for seed, ell_res in cross_corr_ell_all.items():
                for z_obs, res in ell_res.items():
                    if not (z_low <= z_obs < z_high):
                        continue
                    ell   = res['ell_from_k']
                    D_ell = res['D_cross_ell_uK_mK']
                    D_err = res['D_cross_ell_uK_mK_err_total']
                    idx   = np.argmin(np.abs(ell - ell_target))
                    if np.isfinite(D_ell[idx]) and np.isfinite(D_err[idx]):
                        D_vals.append(D_ell[idx])
                        D_errs.append(D_err[idx])

            if len(D_vals) >= 2:
                D_mean = np.mean(D_vals)
                # Error: scatter across seed+redshift samples in bin
                D_std  = np.std(D_vals) / np.sqrt(len(D_vals))
                D_binned.append(D_mean)
                D_err_binned.append(D_std)
                z_used.append(z_bin_centers[j])

        if len(D_binned) > 0:
            z_plot     = np.array(z_used)
            D_plot     = np.array(D_binned)
            D_err_plot = np.array(D_err_binned)
            print(f"    Binned points: {len(D_plot)}")

            ax.errorbar(z_plot, D_plot, yerr=D_err_plot,
                        color=colors_ell[i], lw=2.5, alpha=0.8,
                        marker='o', markersize=6,
                        capsize=4, capthick=1.5,
                        label=f'$\\ell$={ell_target}')
            ax.fill_between(z_plot, D_plot - D_err_plot, D_plot + D_err_plot,
                            color=colors_ell[i], alpha=0.2)

    ax.set_xlabel(r'Redshift $z$', fontsize=14)
    ax.set_ylabel(r'$D_\ell$ [kSZ$^2$-21cm] (μK$^2$·mK)', fontsize=14)
    ax.set_yscale('symlog', linthresh=1e-2)
    ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
    ax.legend(fontsize=14, loc='best', framealpha=0.9)
    ax.invert_xaxis()
    ax.grid(True, alpha=0.3, ls='--')
    ax.text(0.05, 0.95,
            f'Binned Δz ≈ {np.diff(z_bins).mean():.1f} | '
            f'{len(z_bins)-1} bins | {N_SEEDS} seeds',
            transform=ax.transAxes, fontsize=11,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plot_name = "kSZ2_21cm_cross_Dl_vs_z_fixed_ell_BINNED"
    fig.savefig(f"{plot_dir_save}/{plot_name}.pdf", bbox_inches='tight')
    ax.set_title(r'kSZ$^2$-21cm Cross-Power $D_\ell$ Evolution (Binned, Seed-Averaged)',
                 fontsize=16, fontweight='bold')
    fig.savefig(f"{plot_dir_save}/{plot_name}.png", dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_name}")
    plt.close(fig)

    # ==========================================================================
    # PLOT 2: Correlation Coefficient r vs z — binned AND averaged across seeds
    # ==========================================================================

    print(f"\n=== PLOT 2: r vs z (Binned + Seed-Averaged) ===")

    fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

    for i, ell_target in enumerate(ell_targets):
        print(f"  Processing r(ℓ={ell_target})...")

        r_binned = []
        r_err_binned = []
        z_used   = []

        for j in range(len(z_bins) - 1):
            z_low, z_high = z_bins[j], z_bins[j+1]
            r_vals = []

            for seed, ell_res in cross_corr_ell_all.items():
                for z_obs, res in ell_res.items():
                    if not (z_low <= z_obs < z_high):
                        continue
                    ell = res['ell_from_k']
                    r   = res['r_cross']
                    idx = np.argmin(np.abs(ell - ell_target))
                    if (np.isfinite(r[idx]) and not np.isnan(r[idx])
                            and np.abs(r[idx]) < 1.5):
                        r_vals.append(r[idx])

            if len(r_vals) >= 2:
                r_binned.append(np.mean(r_vals))
                r_err_binned.append(np.std(r_vals) / np.sqrt(len(r_vals)))
                z_used.append(z_bin_centers[j])

        if len(r_binned) > 0:
            z_plot     = np.array(z_used)
            r_plot     = np.array(r_binned)
            r_err_plot = np.array(r_err_binned)
            print(f"    Binned points: {len(r_plot)}")

            ax.errorbar(z_plot, r_plot, yerr=r_err_plot,
                        color=colors_ell[i], lw=2.5, alpha=0.8,
                        marker='o', markersize=6,
                        capsize=4, capthick=1.5,
                        label=f'$\\ell$={ell_target}')
            ax.fill_between(z_plot, r_plot - r_err_plot, r_plot + r_err_plot,
                            color=colors_ell[i], alpha=0.2)

    # Ionization fraction markers from reference seed
    z_nodes_sorted = ref_lc.node_redshifts[::-1]
    x_e_nodes      = 1.0 - ref_lc.global_xH[::-1]

    for xe_val in [0.2, 0.5, 0.9]:
        z_xe = np.interp(xe_val, x_e_nodes[::-1], z_nodes_sorted[::-1])
        ax.axvline(z_xe, color='gray', ls=':', lw=1, alpha=0.5)
        ax.text(z_xe, 1.05, f'$x_e$={xe_val:.1f}',
                rotation=90, ha='right', va='top', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    ax.set_xlabel(r'Redshift $z$', fontsize=14)
    ax.set_ylabel(r'Correlation Coefficient $r$', fontsize=14)
    ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
    ax.set_ylim(-1.2, 1.2)
    ax.legend(fontsize=14, loc='best', framealpha=0.9)
    ax.invert_xaxis()
    ax.grid(True, alpha=0.3, ls='--')
    ax.text(0.05, 0.05,
            f'Binned Δz ≈ {np.diff(z_bins).mean():.1f} | '
            f'{len(z_bins)-1} bins | {N_SEEDS} seeds',
            transform=ax.transAxes, fontsize=11,
            verticalalignment='bottom',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plot_name = "kSZ2_21cm_cross_r_vs_z_fixed_ell_BINNED"
    fig.savefig(f"{plot_dir_save}/{plot_name}.pdf", bbox_inches='tight')
    ax.set_title(r'kSZ$^2$-21cm Correlation Coefficient (Binned, Seed-Averaged)',
                 fontsize=16, fontweight='bold')
    fig.savefig(f"{plot_dir_save}/{plot_name}.png", dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_name}")
    plt.close(fig)

    # ==========================================================================
    # PLOT 3: D_ℓ vs ℓ at selected x_e — averaged across seeds
    # ==========================================================================

    print(f"\n=== PLOT 3: D_ℓ vs ℓ at selected x_e (Seed-Averaged) ===")

    target_xe      = [0.2, 0.5, 0.9]
    selected_z     = [z_nodes_sorted[np.argmin(np.abs(x_e_nodes - xe))]
                      for xe in target_xe]
    selected_xe    = [x_e_nodes[np.argmin(np.abs(x_e_nodes - xe))]
                      for xe in target_xe]

    print("  Selected redshifts:")
    for z, xe in zip(selected_z, selected_xe):
        print(f"    z={z:.2f}, x_e={xe:.3f}")

    colors_selected = ['blue', 'green', 'red']
    fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

    for i, (z_target, xe) in enumerate(zip(selected_z, selected_xe)):

        # Collect D_ell arrays from all seeds at closest z
        D_ell_seeds = []
        ell_ref     = None

        for seed, ell_res in cross_corr_ell_all.items():
            z_closest = min(ell_res.keys(), key=lambda z: abs(z - z_target))
            if abs(z_closest - z_target) > 0.5:
                continue
            res   = ell_res[z_closest]
            ell   = res['ell_from_k']
            D_ell = res['D_cross_ell_uK_mK']
            valid = ~np.isnan(D_ell) & np.isfinite(D_ell) & (ell > 10)
            if np.sum(valid) > 5:
                if ell_ref is None:
                    ell_ref   = ell
                    valid_ref = valid
                D_ell_seeds.append(D_ell)

        if len(D_ell_seeds) == 0:
            continue

        D_matrix  = np.array(D_ell_seeds)           # (n_seeds, n_k)
        D_mean    = np.nanmean(D_matrix, axis=0)
        D_std     = np.nanstd(D_matrix,  axis=0) / np.sqrt(len(D_ell_seeds))

        valid = valid_ref & ~np.isnan(D_mean)

        ax.errorbar(ell_ref[valid], D_mean[valid],
                    yerr=D_std[valid],
                    color=colors_selected[i], lw=2.5, alpha=0.8,
                    marker='o', markersize=5, capsize=3, capthick=1.5,
                    label=f'z≈{z_target:.1f} ($x_e$={xe:.2f})',
                    errorevery=3)
        ax.fill_between(ell_ref[valid],
                        D_mean[valid] - D_std[valid],
                        D_mean[valid] + D_std[valid],
                        color=colors_selected[i], alpha=0.15)

    ax.set_xlabel(r'Multipole $\ell$', fontsize=14)
    ax.set_ylabel(r'$D_\ell$ [kSZ$^2$-21cm] (μK$^2$·mK)', fontsize=14)
    ax.set_xscale('log')
    ax.axhline(0, color='black', ls='--', lw=1, alpha=0.5)
    ax.legend(fontsize=14, loc='best', framealpha=0.9)
    ax.text(0.02, 0.02,
            f'{N_SEEDS} seeds averaged',
            transform=ax.transAxes, fontsize=11,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plot_name = "kSZ2_21cm_cross_Dl_selected_xe_with_errors"
    fig.savefig(f"{plot_dir_save}/{plot_name}.pdf", bbox_inches='tight')
    ax.set_title(r'kSZ$^2$-21cm Cross-Power at Key Ionization Fractions'
                 r' (Seed-Averaged)',
                 fontsize=16, fontweight='bold')
    fig.savefig(f"{plot_dir_save}/{plot_name}.png", dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_name}")
    plt.close(fig)

    print("\n✓ ALL VISUALIZATION COMPLETE!")
    print(f"  Plots saved to: {plot_dir_save}")
    print(f"  1. {plot_name}.png  ← D_ℓ vs z binned")
    print(f"  2. kSZ2_21cm_cross_r_vs_z_fixed_ell_BINNED.png  ← r vs z binned")
    print(f"  3. kSZ2_21cm_cross_Dl_selected_xe_with_errors.png  ← D_ℓ vs ℓ")

else:
    print("\n✗ Skipping — no cross_corr_results_all available")

print("\n" + "="*70)


VISUALIZING kSZ²-21cm CROSS-CORRELATION POWER SPECTRA
Final plots directory exists: 18March2026_kSZ2_21cm/plots/plot_final_cell

=== CONVERTING TO ℓ-SPACE WITH ERROR PROPAGATION ===
Converted 155 redshifts to ℓ-space

=== GENERATING D_ℓ EVOLUTION PLOT WITH REDSHIFT BINNING ===
Redshift binning: 30 bins
Bin width: Δz ≈ 0.50

Processing ℓ = 500...
  Binned data points: 30

Processing ℓ = 1000...
  Binned data points: 30

Processing ℓ = 3000...
  Binned data points: 30

✓ Saved: kSZ2_21cm_cross_Dl_vs_z_fixed_ell_BINNED

=== GENERATING CORRELATION COEFFICIENT r vs z PLOT WITH BINNING ===

Processing r(ℓ=500)...
  Binned data points: 30

Processing r(ℓ=1000)...
  Binned data points: 30

Processing r(ℓ=3000)...
  Binned data points: 30

✓ Saved: kSZ2_21cm_cross_r_vs_z_fixed_ell_BINNED

=== GENERATING SELECTED REDSHIFTS PLOT ===
Selected redshifts for labeled plot:
  z=9.78, x_e=0.205
  z=8.02, x_e=0.510
  z=6.40, x_e=0.903
✓ Saved: kSZ2_21cm_cross_Dl_selected_xe_with_errors

✓ ALL VISUALIZA

In [ ]:
# =============================================================================
# CELL 8c: Redshift Evolution of 21cm Auto Power Spectrum (All Seeds)
# Following Cell 8b plotting conventions
# =============================================================================

print("\n" + "="*70)
print("VISUALIZING 21cm AUTO POWER SPECTRUM REDSHIFT EVOLUTION")
print("="*70)

# Create subdirectory if needed
plot_dir_final = f"{plot_dir}/plot_final_cell"
if not os.path.exists(plot_dir_final):
    os.makedirs(plot_dir_final)
plot_dir_save = plot_dir_final

# ==========================================================================
# Load cross_corr_results_all from cache if not in memory
# ==========================================================================
if 'cross_corr_results_all' not in dir() or len(cross_corr_results_all) == 0:
    print("cross_corr_results_all not in memory → loading from Cell 7 cache")
    cross_corr_results_all = {}
    for seed in RANDOM_SEEDS:
        cc_cache = f"{cache_dir}/seed_{seed}/cross_corr_seed{seed}.npy"
        if os.path.exists(cc_cache):
            cross_corr_results_all[seed] = np.load(
                cc_cache, allow_pickle=True).item()
            print(f"  ✓ Loaded seed {seed} "
                  f"({len(cross_corr_results_all[seed])} redshifts)")
        else:
            print(f"  ✗ No cache found for seed {seed}")
    print(f"  Loaded {len(cross_corr_results_all)}/{N_SEEDS} seeds")
else:
    print(f"cross_corr_results_all already in memory "
          f"({len(cross_corr_results_all)} seeds)")

if len(cross_corr_results_all) > 0:

    from astropy.cosmology import FlatLambdaCDM
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors

    cosmo = FlatLambdaCDM(H0=67.77, Om0=0.3086)

    # ==========================================================================
    # Convert P_T21(k) → D_ℓ^{21} for all seeds
    # ==========================================================================

    print(f"\n=== CONVERTING 21cm AUTO POWER TO ℓ-SPACE (ALL SEEDS) ===")

    # {seed: {z_obs: {ell, D_T21_ell, D_T21_ell_err}}}
    auto_T21_ell_all = {}

    for seed, ccr in cross_corr_results_all.items():
        auto_T21_ell_results = {}

        for z_obs in sorted(ccr.keys()):
            results      = ccr[z_obs]
            D_A_Mpc      = float(cosmo.angular_diameter_distance(z_obs).value)
            chi_comoving = float(cosmo.comoving_distance(z_obs).value)

            k_centers  = results['k_centers']
            P_T21      = results['P_T21_1d']
            n_modes    = results['n_modes']

            ell_from_k = k_centers * chi_comoving / 0.67
            C_T21_ell  = P_T21 * 0.67**2 / D_A_Mpc**2
            D_T21_ell  = ell_from_k * (ell_from_k + 1) * C_T21_ell / (2 * np.pi)

            with np.errstate(divide='ignore', invalid='ignore'):
                err_frac = np.where(n_modes > 0, 1.0 / np.sqrt(n_modes), np.nan)
            D_T21_ell_err = np.abs(D_T21_ell) * err_frac

            auto_T21_ell_results[z_obs] = {
                'ell_from_k'   : ell_from_k,
                'D_T21_ell'    : D_T21_ell,
                'D_T21_ell_err': D_T21_ell_err,
            }

        auto_T21_ell_all[seed] = auto_T21_ell_results

    print(f"Converted {len(auto_T21_ell_all)} seeds to ℓ-space")

    # ==========================================================================
    # Average D_ℓ^{21} across seeds at each redshift
    # ==========================================================================

    ref_seed = list(auto_T21_ell_all.keys())[0]
    all_z    = sorted(auto_T21_ell_all[ref_seed].keys())

    auto_T21_ell_averaged = {}   # {z_obs: {ell, D_mean, D_err}}

    for z_obs in all_z:
        D_seeds = []
        ell_ref = None

        for seed, res_dict in auto_T21_ell_all.items():
            if z_obs not in res_dict:
                continue
            res   = res_dict[z_obs]
            ell   = res['ell_from_k']
            D_ell = res['D_T21_ell']
            if ell_ref is None:
                ell_ref = ell
            D_seeds.append(D_ell)

        if len(D_seeds) == 0:
            continue

        D_matrix = np.array(D_seeds)
        D_mean   = np.nanmean(D_matrix, axis=0)
        D_std    = np.nanstd(D_matrix,  axis=0) / np.sqrt(len(D_seeds))

        auto_T21_ell_averaged[z_obs] = {
            'ell_from_k'   : ell_ref,
            'D_T21_ell'    : D_mean,
            'D_T21_ell_err': D_std,
        }

    print(f"Averaged over {len(auto_T21_ell_all)} seeds "
          f"at {len(auto_T21_ell_averaged)} redshifts")

    # ==========================================================================
    # PLOT: D_ℓ^{21} vs ℓ — all redshifts, color-coded by z, seed-averaged
    # ==========================================================================

    print(f"\n=== GENERATING 21cm AUTO POWER SPECTRUM PLOT ===")

    z_all  = np.array(sorted(auto_T21_ell_averaged.keys()))
    norm   = mcolors.Normalize(vmin=z_all.min(), vmax=z_all.max())
    cmap   = cm.plasma

    fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

    for z_obs in z_all:
        res   = auto_T21_ell_averaged[z_obs]
        ell   = res['ell_from_k']
        D_ell = res['D_T21_ell']
        D_err = res['D_T21_ell_err']

        valid = (~np.isnan(D_ell) & np.isfinite(D_ell)
                 & (ell > 10) & (D_ell > 0))
        if np.sum(valid) < 3:
            continue

        color = cmap(norm(z_obs))
        ax.plot(ell[valid], D_ell[valid],
                color=color, lw=1.5, alpha=0.75)
        ax.fill_between(ell[valid],
                        D_ell[valid] - D_err[valid],
                        D_ell[valid] + D_err[valid],
                        color=color, alpha=0.12)

    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r'Redshift $z$', fontsize=13)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'Multipole $\ell$', fontsize=14)
    ax.set_ylabel(r'$D_\ell^{21}\ [\mathrm{mK}^2]$', fontsize=14)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.text(0.05, 0.05,
            f'Shaded: seed-to-seed scatter / √N\n'
            f'{N_SEEDS} seeds averaged\n'
            r'$D_\ell = \ell(\ell+1)C_\ell/2\pi$',
            transform=ax.transAxes, fontsize=11,
            verticalalignment='bottom',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plot_name = "21cm_auto_Dl_vs_ell_redshift_evolution"
    fig.savefig(f"{plot_dir_save}/{plot_name}.pdf", bbox_inches='tight')
    ax.set_title(r'21cm Auto Power Spectrum $D_\ell^{21}$ — Redshift Evolution'
                 f' ({N_SEEDS} seeds)',
                 fontsize=16, fontweight='bold')
    fig.savefig(f"{plot_dir_save}/{plot_name}.png", dpi=300, bbox_inches='tight')
    print(f"✓ Saved: {plot_name}")
    plt.close(fig)

    print("\n✓ 21cm AUTO POWER SPECTRUM PLOT COMPLETE")
    print(f"  Saved to: {plot_dir_save}")

else:
    print("\n✗ Skipping — no cross_corr_results_all available")

print("\n" + "="*70)


VISUALIZING 21cm AUTO POWER SPECTRUM REDSHIFT EVOLUTION

=== CONVERTING 21cm AUTO POWER TO ℓ-SPACE ===
Converted 155 redshifts to ℓ-space

=== GENERATING 21cm AUTO POWER SPECTRUM PLOT ===
✓ Saved: 21cm_auto_Dl_vs_ell_redshift_evolution

✓ 21cm AUTO POWER SPECTRUM PLOT COMPLETE
  Saved to: 18March2026_kSZ2_21cm/plots/plot_final_cell

